# Fusion Benchmark v2

Compares three **fusion levels** for combining the two channels (SER emotion + ASR transcript)
into an anomaly verdict, plus two extra intermediate variants and three mandatory ablation baselines.

**Framing — say this out loud.** This is NOT end-to-end audio. The pipeline reduces audio to
(text, emotion), so what is benchmarked is the **fusion combiner at the symbolic level**.

## What is new in v2

| # | Addition | Why |
|---|---|---|
| 1 | `text_only_finetuned` | Isolates "emotion at the input" from "the encoder got fine-tuned". Without it, `early`'s win is confounded by capacity. |
| 2 | `CLASS_WEIGHTING` run whole-matrix, both settings | `f1_borderline` was exactly 0.000 for several methods — an imbalance collapse, not a finding. |
| 3 | Equal-budget sweep for `late` and `intermediate` | Those two carry the headline comparison, so they must get identical tuning budget. |
| 4 | Post-hoc class-bias correction | Turned out to matter enormously — see the warning below. |
| 5 | `intermediate_film` | Emotion is only ~4% of the concatenated width in `intermediate`; FiLM gives it leverage over every text dimension. |
| 6 | `intermediate_attn` | Emotion queries the transcript. Uses TOKEN-level features, so it sidesteps the CLS bottleneck. |
| 7 | **SER noise model** | The emotion channel was an oracle label. At deployment it comes from a model that is wrong ~14% of the time at the arousal level. |

## Two results from v1 you must not forget

**"Late fusion is worse than its own emotion input" was a calibration artifact.** After post-hoc
bias correction, `late` beat `emotion_only` in all four cells. Bias correction adds ~+0.10 macro-F1
to `late` and ~0.00 to `intermediate_attn`. Report the corrected numbers, or you will publish a
conclusion about fusion that is really a conclusion about decision thresholds.

**`early` vs `intermediate` flips depending on the encoder.** Defensible claim:
`{attn, early} > intermediate > late > single channel`. NOT "attn is best" — on bert/full the
attn-vs-early gap was +0.011 against a pooled seed spread of 0.020, i.e. noise.

## Leakage rules baked into the code
- Only `text` and `gen_emotion` are model inputs.
- `judge_content_risk` / `judge_voice_risk` are **banned** — outputs of the same LLM call that
  produced the label. A 6-cell lookup over them scores 91.5%.
- `event` is **banned** — measured by LLM audit (n=1000) to be wrong for 6.7% [5.2-8.3] of rows.
- Splits are by **`seed_id` group**, never by row: utterances sharing a seed are near-paraphrases.


## 1. Setup

In [ ]:
!pip install -q transformers sentence-transformers scikit-learn --upgrade
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time

# --- EDIT IF YOUR LAYOUT DIFFERS -------------------------------------------
DRIVE_ROOT   = "/content/drive/MyDrive/CLEAR/fusion"
DATASET_PATH = f"{DRIVE_ROOT}/dataset_final.jsonl"
# If the dataset is not on Drive yet, upload it to the Colab file pane and use:
# DATASET_PATH = "/content/dataset_final.jsonl"
# ---------------------------------------------------------------------------

# Embedding caches live on Drive so the second, third and fourth phases of the
# run plan (and any later session) reuse them instead of re-encoding 9,740 rows.
CACHE_DIR     = f"{DRIVE_ROOT}/emb_cache"
TOK_CACHE_DIR = f"{DRIVE_ROOT}/emb_cache_tok"
RUNS_DIR      = f"{DRIVE_ROOT}/runs"
LOCAL_OUT     = "/content/fusion_out"

for d in (CACHE_DIR, TOK_CACHE_DIR, RUNS_DIR, LOCAL_OUT):
    os.makedirs(d, exist_ok=True)

assert os.path.exists(DATASET_PATH), f"dataset not found at {DATASET_PATH}"
print("dataset:", DATASET_PATH, os.path.getsize(DATASET_PATH) // 1024, "KB")
print("drive out:", RUNS_DIR)

## 2. Modules

Written to disk from `%%writefile` cells so the notebook stays self-contained while the code stays
modular and diffable.

In [ ]:
%%writefile common.py
"""the fusion benchmark — shared constants, data loading, splitting and metrics.

LEAKAGE RULE
------------
The only fields allowed as MODEL INPUT are `text` and `gen_emotion`.
`seed_id` is used only for splitting, `uid` only as a cache key, `source_model`
only for the data-variant filter.

Every field in BANNED_FIELDS is forbidden as a feature:

  * `judge_content_risk`, `judge_voice_risk`, `gen_content_risk`,
    `content_risk_seed` — outputs of the same LLM call that produced the
    `anomaly` label. A 6-cell lookup table over
    (judge_voice_risk, judge_content_risk) scores 91.5% on the full dataset.
    That is label leakage, not a result.
  * `event`, `profile`, `target_emotion` — seed metadata. `event` was measured
    by LLM audit (n=1000) to be wrong for 6.7% [5.2-8.3] of rows, because the
    generator drifted off its seeded scenario.
  * `reason`, `judge_*`, `anomaly_votes` — annotation metadata about the label.
  * `source_model` — which LLM wrote the utterance; a style shortcut.

This module has no torch/transformers dependency so it stays importable on a
CPU-only machine.
"""

from __future__ import annotations

import json
import random

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

LABELS: list[str] = ["normal", "borderline", "anomaly"]
EMOTIONS: list[str] = ["neutral", "confusion", "fear", "panic", "urgency", "distress"]

ENCODER_IDS: dict[str, str] = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",  # 384-dim
    "bert": "bert-base-uncased",  # 768-dim, CLS pooling
}

VARIANTS: list[str] = ["full", "filtered"]

# Global, all-or-nothing switch for class-weighted training loss. Every
# trainable method (run_late, run_intermediate, run_early,
# run_text_only_finetuned, run_intermediate_film, run_intermediate_attn) and
# both LogisticRegression baselines (emotion_only, text_only) honour this flag
# via `class_weights()` / `class_weight="balanced"`. The driver must run the
# WHOLE method matrix with this flag at one setting and report both settings
# side by side — never mix weighted and unweighted rows in the same table,
# or a "tuned vs untuned" artefact would masquerade as a fusion-level finding.
CLASS_WEIGHTING: bool = False

# Utterances from these two generators had a measured event-drift rate of
# 18.1% and 11.4%, vs ~1% for deepseek-v4. The "filtered" variant drops them
# as a robustness ablation.
FILTERED_OUT_MODELS: set[str] = {"qwen3.6-flash", "qwen3.5-flash"}

BANNED_FIELDS: set[str] = {
    "judge_content_risk",
    "judge_voice_risk",
    "gen_content_risk",
    "content_risk_seed",
    "event",
    "profile",
    "target_emotion",
    "reason",
    "judge_model",
    "judge_count",
    "judge_agreement",
    "judge_models",
    "anomaly_votes",
    "source_model",
}

_LABEL_TO_IDX: dict[str, int] = {name: i for i, name in enumerate(LABELS)}
_EMOTION_TO_IDX: dict[str, int] = {name: i for i, name in enumerate(EMOTIONS)}

# --- emotion regime ---------------------------------------------------------
# Which emotion channel the fusion methods see. Read LIVE off this module
# (`_common.EMOTION_REGIME`) — never `from common import
# EMOTION_REGIME`, which would freeze the value at import time and silently
# keep using a stale setting when the driver switches regimes.
#
#   "oracle"   — train on the dataset's `gen_emotion`, test on it too.
#                The upper bound. Not achievable in deployment.
#   "ser_test" — train on the oracle label, test on SER-realistic emotion.
#                Measures how far the current approach falls when it meets the
#                real SER. No retraining, so this is pure robustness measurement.
#   "ser_both" — train AND test on SER-realistic emotion. The fix: the combiner
#                learns to be robust to the SER's measured error pattern.
#
# Compare "ser_test" against "ser_both" to see whether training under noise
# closes the deployment gap. Both share the same test condition, so the
# comparison is fair; "oracle" is the ceiling above them.
EMOTION_REGIMES: list[str] = ["oracle", "ser_test", "ser_both"]
EMOTION_REGIME: str = "oracle"

# --- checkpoint sink --------------------------------------------------------
# When the driver sets this to a dict, each trainable method deposits its
# best-by-val state dict here instead of discarding it, so the driver can decide
# what is worth persisting. Left as None by default: keeping every checkpoint
# would mean ~48 fine-tuned encoders at ~440 MB each (>20 GB), which is useless
# — the app needs ONE deployable model, not forty-eight. Policy applied by the
# driver: keep every small head (they are kilobytes), and of the fine-tuned
# encoders keep only the best val macro-F1 per (encoder, variant).
CHECKPOINT_SINK: dict | None = None

# --- cache locations --------------------------------------------------------
# `run_intermediate_attn` builds its own token-level embedding cache rather than
# receiving one, so the location has to be settable from outside or the cache
# lands on ephemeral Colab local disk and gets rebuilt (~2 GB, several minutes)
# every session. Read LIVE off this module, same discipline as the flags above.
TOKEN_CACHE_DIR: str = "/content/emb_cache_tok"

# --- fine-tuning budget -----------------------------------------------------
# Epochs for the two fine-tuning methods (`early`, `text_only_finetuned`).
# Read LIVE so a smoke test can drop it to 1 and still exercise the real code
# path — the point of a pre-flight check is to prove the expensive branch RUNS,
# not to get a good score from it. 4 is the value the reported results used.
MAX_FINETUNE_EPOCHS: int = 4


def offer_checkpoint(tag: str, state_dict, val_f1: float, meta: dict | None = None) -> None:
    """Hand a trained model to the driver, if it asked for one.

    Deliberately a no-op when CHECKPOINT_SINK is None so that sweeps — which
    train hundreds of throwaway models — cost nothing.
    """
    if CHECKPOINT_SINK is None:
        return
    CHECKPOINT_SINK[tag] = {
        "state_dict": state_dict,
        "val_f1": float(val_f1),
        "meta": dict(meta or {}),
    }


def set_seed(seed: int) -> None:
    """Seed every RNG in play. Torch is optional so this module stays light."""
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


def load_rows(path: str, variant: str = "full") -> list[dict]:
    """Parse the JSONL dataset. `variant="filtered"` drops the drift-prone models."""
    if variant not in VARIANTS:
        raise ValueError(f"unknown variant {variant!r}, expected one of {VARIANTS}")

    rows: list[dict] = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            if variant == "filtered" and row.get("source_model") in FILTERED_OUT_MODELS:
                continue
            rows.append(row)

    if not rows:
        raise ValueError(f"no rows loaded from {path!r} for variant {variant!r}")
    return rows


def make_splits(
    rows: list[dict],
    split_seed: int = 42,
    val_frac: float = 0.15,
    test_frac: float = 0.15,
    seed_universe: list[int] | None = None,
) -> dict[str, list[dict]]:
    """Split on `seed_id` GROUPS, not rows.

    Utterances sharing a seed_id are near-paraphrases of one another (same
    profile/event/emotion, different generator). A row-level split would put
    paraphrases of the same scenario on both sides and the model would score
    high by memorisation. Fractions apply to the number of seed groups.

    Depends only on `split_seed`, never on a per-run seed, so every method,
    encoder and variant is evaluated on exactly the same held-out scenarios.

    `seed_universe` pins the train/val/test assignment to a fixed set of
    seed_ids. The driver MUST pass the full dataset's seed_ids here for both
    data variants: the "filtered" variant contains fewer unique seeds, so
    letting it shuffle its own universe would give it a different test set and
    confound the ablation — a full-vs-filtered gap would then be split noise
    rather than a data-quality effect. With the universe pinned, the filtered
    splits are strict row-subsets of the full ones.
    """
    if not 0 < val_frac + test_frac < 1:
        raise ValueError("val_frac + test_frac must be in (0, 1)")

    present = {row["seed_id"] for row in rows}
    if seed_universe is None:
        unique_seeds = sorted(present)
    else:
        unique_seeds = sorted(set(seed_universe))
        missing = present - set(unique_seeds)
        if missing:
            raise ValueError(
                f"{len(missing)} seed_id(s) in rows are absent from seed_universe, "
                f"e.g. {sorted(missing)[:5]}"
            )
    shuffled = list(unique_seeds)
    random.Random(split_seed).shuffle(shuffled)

    n = len(shuffled)
    n_test = int(round(n * test_frac))
    n_val = int(round(n * val_frac))

    test_seeds = set(shuffled[:n_test])
    val_seeds = set(shuffled[n_test : n_test + n_val])

    splits: dict[str, list[dict]] = {"train": [], "val": [], "test": []}
    for row in rows:  # preserves dataset order within each split
        sid = row["seed_id"]
        if sid in test_seeds:
            splits["test"].append(row)
        elif sid in val_seeds:
            splits["val"].append(row)
        else:
            splits["train"].append(row)
    return splits


def assert_no_seed_overlap(splits: dict[str, list[dict]]) -> None:
    """Raise if any seed_id appears in more than one split."""
    groups = {name: {row["seed_id"] for row in rows} for name, rows in splits.items()}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        shared = groups[a] & groups[b]
        assert not shared, f"seed_id leak between {a} and {b}: {sorted(shared)[:10]}"


def y_of(rows: list[dict]) -> np.ndarray:
    """Target labels as int indices into LABELS."""
    return np.array([_LABEL_TO_IDX[row["anomaly"]] for row in rows], dtype=np.int64)


def emotion_onehot(rows: list[dict]) -> np.ndarray:
    """(N, 6) float32 one-hot of `gen_emotion`."""
    out = np.zeros((len(rows), len(EMOTIONS)), dtype=np.float32)
    for i, row in enumerate(rows):
        out[i, _EMOTION_TO_IDX[row["gen_emotion"]]] = 1.0
    return out


def emotion_idx(rows: list[dict]) -> np.ndarray:
    """(N,) int indices of `gen_emotion`.

    Kept for backward compatibility and for the hard-label path. New code should
    prefer `emotion_features`, which also covers the soft/noisy regimes.
    """
    return np.array([_EMOTION_TO_IDX[row["gen_emotion"]] for row in rows], dtype=np.int64)


def emotion_features(
    rows: list[dict],
    split_name: str,
    seed: int,
    regime: str | None = None,
) -> np.ndarray:
    """(N, 6) float32 emotion channel, honouring the active EMOTION_REGIME.

    This is the single place every fusion method gets its emotion input from, so
    a regime switch applies uniformly and cannot be applied to some methods but
    not others.

    Under "oracle" the result is a plain one-hot, which means the models behave
    exactly as they did before this function existed — `nn.Linear(6, k,
    bias=False)` applied to a one-hot is mathematically identical to
    `nn.Embedding(6, k)`, so previously reported hard-label numbers remain valid.

    Args:
        rows: the split's rows.
        split_name: "train", "val" or "test". Decides whether noise applies:
            "ser_test" noises only the test split (the model was trained on the
            oracle label, so val must stay oracle too or model selection would
            not match how it was trained).
        seed: the per-run seed. SER errors are re-drawn per seed, so the spread
            across seeds is a real error bar on the robustness estimate.
        regime: override; defaults to the live module-level EMOTION_REGIME.
    """
    active = EMOTION_REGIME if regime is None else regime
    if active not in EMOTION_REGIMES:
        raise ValueError(f"unknown emotion regime {active!r}, expected {EMOTION_REGIMES}")
    if split_name not in ("train", "val", "test"):
        raise ValueError(f"unknown split name {split_name!r}")

    noisy = active == "ser_both" or (active == "ser_test" and split_name == "test")
    if not noisy:
        return emotion_onehot(rows)

    # Imported lazily: ser_noise imports EMOTIONS from this module,
    # so a top-level import here would be circular.
    from ser_noise import simulate

    # Offset the seed per split so train/val/test do not receive correlated
    # error draws.
    offset = {"train": 0, "val": 1_000, "test": 2_000}[split_name]
    return simulate([row["gen_emotion"] for row in rows], seed + offset)


def class_weights(y_train: np.ndarray) -> np.ndarray:
    """Inverse-frequency class weights over LABELS, normalised to mean 1.

    Used to build `nn.CrossEntropyLoss(weight=...)` for every trainable
    method when `CLASS_WEIGHTING` is on, and as the numeric basis for
    `class_weight="balanced"` semantics for the LogisticRegression baselines
    (those call sklearn's own "balanced" option directly; this function is
    for the torch methods, which have no such built-in).

    `borderline` is 12.4% of train in the observed run — this makes the
    minority class(es) worth proportionally more in the loss, without ever
    touching val/test, so it addresses the f1_borderline==0.0 collapse
    without resampling.
    """
    counts = np.bincount(y_train, minlength=len(LABELS)).astype(np.float64)
    if (counts == 0).any():
        # A class absent from train would divide by zero; treat it as
        # maximally rare rather than crash, though a model can't learn a
        # class it never sees regardless of loss weighting.
        counts = np.where(counts == 0, 1.0, counts)
    inv = 1.0 / counts
    inv = inv * (len(LABELS) / inv.sum())  # normalise so mean weight == 1
    return inv.astype(np.float32)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Accuracy, macro-F1, per-class F1 and the 3x3 confusion matrix (JSON-safe).

    `labels=[0, 1, 2]` is pinned explicitly: a degenerate model that never
    predicts `borderline` would otherwise return a 2-element per-class array
    and silently misalign the f1_* columns.
    """
    per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_normal": float(per_class[0]),
        "f1_borderline": float(per_class[1]),
        "f1_anomaly": float(per_class[2]),
        "confusion": [[int(v) for v in row] for row in cm],
    }


def make_result(
    method: str,
    encoder: str,
    variant: str,
    seed: int,
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict:
    """One flat, JSON-serialisable record for the results table."""
    return {
        "method": method,
        "encoder": encoder,
        "variant": variant,
        "seed": int(seed),
        **compute_metrics(y_true, y_pred),
    }


def self_test(path: str) -> None:
    """Smoke test: split integrity and label distribution. Run this first."""
    from collections import Counter

    universe = sorted({row["seed_id"] for row in load_rows(path, "full")})
    test_uids: dict[str, set[int]] = {}

    for variant in VARIANTS:
        rows = load_rows(path, variant=variant)
        splits = make_splits(rows, seed_universe=universe)
        assert_no_seed_overlap(splits)
        test_uids[variant] = {row["uid"] for row in splits["test"]}

        total = sum(len(v) for v in splits.values())
        assert total == len(rows), f"row count changed: {total} != {len(rows)}"
        uids = [row["uid"] for split in splits.values() for row in split]
        assert len(set(uids)) == len(uids), "duplicate uid across splits"

        print(f"[{variant}] {len(rows)} rows, {len({r['seed_id'] for r in rows})} seeds")
        for name in ("train", "val", "test"):
            part = splits[name]
            dist = Counter(row["anomaly"] for row in part)
            share = {k: round(v / len(part), 3) for k, v in sorted(dist.items())}
            print(
                f"  {name:5s} {len(part):5d} rows  "
                f"{len({r['seed_id'] for r in part}):4d} seeds  {share}"
            )

    assert test_uids["filtered"] <= test_uids["full"], (
        "filtered test set is not a subset of the full test set — the two "
        "variants are being evaluated on different scenarios"
    )
    print("filtered test set is a strict row-subset of the full test set")
    print("self_test passed")

In [ ]:
%%writefile ser_noise.py
"""SER noise model — makes the fusion combiner face the emotion channel it will
actually be deployed with, instead of the oracle label.

WHY THIS EXISTS
---------------
Every fusion method in this benchmark consumes `gen_emotion`, a HARD label taken
straight from the dataset. At deployment the emotion does not come from a
dataset — it comes from the SER model, which is wrong a substantial fraction of
the time. And in THIS task the error does not degrade gracefully: the anomaly
label is defined as a mismatch between voice arousal and content severity, so a
flipped arousal reading does not merely add noise, it INVERTS the verdict.

MEASURED GROUND TRUTH (not invented)
------------------------------------
From `_Staj/meeting/D3_voice_channel.png` — the SER voice-risk confusion matrix
on the held-out, speaker-independent validation split, n=902:

                        true=high   true=low
    SER pred = high        512         22       (534)
    SER pred = low         104        264       (368)
                           616        286       (902)

This reconciles exactly with the three numbers reported for the SER channel
(accuracy 86.03%, precision 95.88%, recall 83.12%), so the matrix is the real
measured one rather than a reconstruction.

WHY A BINARY MATRIX IS ENOUGH
-----------------------------
The 6-class SER confusion matrix was not recoverable. It turns out not to
matter much: `judge_voice_risk` in the fusion dataset is a 99.9% deterministic
recode of `gen_emotion` into two arousal buckets
(neutral/confusion -> low, fear/panic/urgency/distress -> high), so the axis
that actually drives the anomaly label IS this binarisation — and that is
precisely what the n=902 matrix measures.

WHAT IS MEASURED VS ASSUMED — state this honestly on any slide
--------------------------------------------------------------
  MEASURED (n=902): the arousal flip rates and their posteriors.
  ASSUMED:          which specific emotion inside a bucket the SER would name.
                    We distribute that mass by the dataset's own within-bucket
                    emotion prior. Nothing in the anomaly label depends on this
                    choice, since the label is driven by the bucket.

IS COLLAPSING TO THE BUCKET A LOSS? MEASURED: NO.
-------------------------------------------------
This model produces only two distinct soft vectors (one per predicted bucket),
so it reduces the emotion channel to a single bit. That is a fair worry, so it
was tested rather than assumed. Predicting `anomaly` on the held-out test split
from the emotion channel alone:

    6-way oracle one-hot     macro-F1 = 0.4111
    2-way arousal bucket     macro-F1 = 0.4111

Identical. The fine-grained emotion identity carries no label-relevant
information beyond its arousal bucket, which is the expected consequence of
`judge_voice_risk` being a binary recode in the first place. So the simple
bucket-level noise model loses nothing, and modelling within-bucket confusion
(which would require an extra independence assumption on top of the 6-class UA)
would add assumptions for no measurable benefit. Fewer assumptions wins.

THE TRAP THIS MODULE AVOIDS
---------------------------
The naive implementation is to hand every high-arousal row the same "expected"
soft vector [0.831 on high, 0.169 on low]. That destroys NO information: every
high row still presents identically, so a model can invert the transform
perfectly and recover the oracle label. It would look like a robustness test
while measuring nothing.

Instead we SAMPLE the predicted bucket per row from the measured likelihoods
(so 16.9% of genuinely high-arousal rows really do present as low), and only
then build a soft vector around the sampled outcome using the measured
posterior error for that prediction. The information loss is real.
"""

from __future__ import annotations

import numpy as np

from common import EMOTIONS

# --- the measured confusion matrix, n=902 -----------------------------------
SER_CONFUSION = {
    ("high", "high"): 512,  # (predicted, true)
    ("high", "low"): 22,
    ("low", "high"): 104,
    ("low", "low"): 264,
}
SER_CONFUSION_N = 902
SER_CONFUSION_SOURCE = "_Staj/meeting/D3_voice_channel.png (held-out val, n=902)"

# Arousal bucket of each of the SER's six classes. This is the same mapping
# that `judge_voice_risk` follows in the dataset (verified: 99.9% agreement).
AROUSAL_BUCKET = {
    "neutral": "low",
    "confusion": "low",
    "fear": "high",
    "panic": "high",
    "urgency": "high",
    "distress": "high",
}
BUCKETS = ("low", "high")

# Emotion counts in dataset_final.jsonl, used as the within-bucket prior.
EMOTION_PRIOR_COUNTS = {
    "neutral": 2267,
    "confusion": 1978,
    "fear": 1616,
    "panic": 1399,
    "urgency": 1386,
    "distress": 1094,
}


def _col_totals() -> dict[str, int]:
    """Column totals = how many rows of each TRUE bucket the matrix saw."""
    return {
        true: sum(v for (_, t), v in SER_CONFUSION.items() if t == true)
        for true in BUCKETS
    }


def _row_totals() -> dict[str, int]:
    """Row totals = how many times the SER PREDICTED each bucket."""
    return {
        pred: sum(v for (p, _), v in SER_CONFUSION.items() if p == pred)
        for pred in BUCKETS
    }


def likelihoods() -> dict[str, dict[str, float]]:
    """P(SER predicts `pred` | true bucket is `true`), column-normalised.

    This is the information-destroying step: it is what decides that ~16.9% of
    genuinely high-arousal utterances will be presented to the fusion model as
    low arousal.
    """
    col = _col_totals()
    return {
        true: {pred: SER_CONFUSION[(pred, true)] / col[true] for pred in BUCKETS}
        for true in BUCKETS
    }


def posteriors() -> dict[str, dict[str, float]]:
    """P(true bucket is `true` | SER predicted `pred`), row-normalised.

    Used to shape the soft vector. Note the strong asymmetry this exposes:
    a "high" prediction is wrong only ~4% of the time, but a "low" prediction
    is wrong ~28% of the time. A well-calibrated downstream model should
    therefore trust "high" far more than "low".
    """
    row = _row_totals()
    return {
        pred: {true: SER_CONFUSION[(pred, true)] / row[pred] for true in BUCKETS}
        for pred in BUCKETS
    }


def within_bucket_prior() -> dict[str, np.ndarray]:
    """For each bucket, a distribution over the 6 emotions summing to 1.

    Emotions outside the bucket get exactly 0. This is the ASSUMED part of the
    noise model — see the module docstring.
    """
    out: dict[str, np.ndarray] = {}
    for bucket in BUCKETS:
        w = np.array(
            [
                EMOTION_PRIOR_COUNTS[e] if AROUSAL_BUCKET[e] == bucket else 0.0
                for e in EMOTIONS
            ],
            dtype=np.float64,
        )
        out[bucket] = (w / w.sum()).astype(np.float32)
    return out


def summary() -> str:
    """Human-readable report of the noise model, for a notebook cell or slide."""
    lk, po = likelihoods(), posteriors()
    lines = [
        f"SER noise model — source: {SER_CONFUSION_SOURCE}",
        "",
        "  MEASURED likelihoods (what the SER does to a true bucket):",
        f"    P(pred=low  | true=high) = {lk['high']['low']:.4f}   <- miss",
        f"    P(pred=high | true=low ) = {lk['low']['high']:.4f}   <- false alarm",
        f"    misses are {lk['high']['low'] / lk['low']['high']:.1f}x more likely "
        f"than false alarms",
        "",
        "  MEASURED posteriors (how much to trust a prediction):",
        f"    P(true=low  | pred=high) = {po['high']['low']:.4f}",
        f"    P(true=high | pred=low ) = {po['low']['high']:.4f}",
        "",
        "  Consequence for the two anomaly directions:",
        f"    calm voice + severe content   -> survives {1 - lk['low']['high']:.1%} of the time (robust)",
        f"    alarmed voice + trivial content -> survives {1 - lk['high']['low']:.1%} of the time (fragile)",
    ]
    return "\n".join(lines)


def simulate(
    true_emotions: list[str],
    seed: int,
    hard: bool = False,
) -> np.ndarray:
    """Turn oracle emotion labels into SER-realistic emotion features.

    Args:
        true_emotions: the oracle `gen_emotion` string for each row.
        seed: RNG seed. Different run seeds draw different SER errors, so the
            spread across seeds is itself a meaningful error bar on the
            robustness estimate — do not fix this to a constant.
        hard: if True return a one-hot of the sampled emotion instead of a soft
            distribution. Useful as an ablation isolating "soft vs hard" from
            "noisy vs oracle".

    Returns:
        (N, 6) float32, each row summing to 1, column order = EMOTIONS.
    """
    rng = np.random.default_rng(seed)
    lk = likelihoods()
    po = posteriors()
    prior = within_bucket_prior()
    other = {"low": "high", "high": "low"}

    out = np.zeros((len(true_emotions), len(EMOTIONS)), dtype=np.float32)

    for i, emo in enumerate(true_emotions):
        true_bucket = AROUSAL_BUCKET[emo]

        # Step 1 — sample what the SER would have PREDICTED. This is the step
        # that genuinely destroys information, and every number in it is
        # measured from the n=902 matrix.
        p_high = lk[true_bucket]["high"]
        pred_bucket = "high" if rng.random() < p_high else "low"

        if hard:
            # Collapse to a single emotion drawn from the predicted bucket.
            out[i] = rng.multinomial(1, prior[pred_bucket]).astype(np.float32)
            continue

        # Step 2 — shape the soft vector using the measured posterior for that
        # prediction, so the vector carries the SER's real uncertainty.
        conf = po[pred_bucket][pred_bucket]  # mass on the predicted bucket
        out[i] = conf * prior[pred_bucket] + (1.0 - conf) * prior[other[pred_bucket]]

    return out


def flip_report(true_emotions: list[str], simulated: np.ndarray) -> dict:
    """How many rows actually had their arousal bucket inverted.

    Compares the oracle bucket against the argmax bucket of the simulated
    features, which is what a downstream hard-label consumer would see.
    """
    counts = {"high->low": 0, "low->high": 0, "unchanged": 0}
    for emo, vec in zip(true_emotions, simulated):
        true_bucket = AROUSAL_BUCKET[emo]
        seen_bucket = AROUSAL_BUCKET[EMOTIONS[int(np.argmax(vec))]]
        if true_bucket == seen_bucket:
            counts["unchanged"] += 1
        else:
            counts[f"{true_bucket}->{seen_bucket}"] += 1
    n = len(true_emotions)
    counts["flipped_frac"] = round((n - counts["unchanged"]) / n, 4)
    counts["n"] = n
    return counts

In [ ]:
%%writefile encoders.py
"""
encoders.py

Encoder abstraction: converts text to embeddings for the fusion benchmark.
Supports "minilm" (sentence-transformers, 384-dim) and "bert" (transformers, 768-dim with CLS pooling).

Models are cached by (encoder_name, device) to avoid reloading during the 12-run matrix.
Embeddings are cached by encoder_name, split name, row count, and SHA1 hash of uid list,
ensuring different data variants (full vs filtered) get separate cache files.

All embeddings are returned as np.float32.
"""

import os
import hashlib
import numpy as np

# Module-level model cache: {(encoder_name, device): model}
# This is the one exception to the "no global mutable state" rule — intentional,
# because model loading is expensive and we run a 12-run matrix (6 methods x 2 encoders x 1 variant = 12 runs).
_model_cache: dict = {}


def embed_texts(texts: list[str], encoder_name: str, device: str,
                batch_size: int = 64) -> np.ndarray:
    """
    Embed a list of texts using the specified encoder.

    Lazy-imports torch/transformers/sentence_transformers inside the function so the module
    can be imported on machines without the deep-learning stack.

    Args:
        texts: list of strings to encode
        encoder_name: "minilm" (sentence-transformers, 384-dim) or "bert" (transformers, 768-dim CLS)
        device: "cpu" or "cuda"
        batch_size: batch size for encoding (default 64)

    Returns:
        np.ndarray of shape (len(texts), D) with dtype float32
        D = 384 for minilm, 768 for bert

    Raises:
        ValueError: if encoder_name is not "minilm" or "bert"
    """
    if encoder_name not in ["minilm", "bert"]:
        raise ValueError(f"Unknown encoder_name: {encoder_name}")

    cache_key = (encoder_name, device)

    if encoder_name == "minilm":
        # Lazy import
        if cache_key not in _model_cache:
            from sentence_transformers import SentenceTransformer
            from common import ENCODER_IDS

            model = SentenceTransformer(ENCODER_IDS["minilm"])
            model.to(device)
            _model_cache[cache_key] = model

        model = _model_cache[cache_key]
        # SentenceTransformer.encode handles batching and returns normalized embeddings
        embeddings = model.encode(texts, batch_size=batch_size,
                                 normalize_embeddings=True, convert_to_numpy=True)
        return embeddings.astype(np.float32)

    elif encoder_name == "bert":
        # Lazy imports
        if cache_key not in _model_cache:
            import torch
            from transformers import AutoTokenizer, AutoModel
            from common import ENCODER_IDS

            tokenizer = AutoTokenizer.from_pretrained(ENCODER_IDS["bert"])
            model = AutoModel.from_pretrained(ENCODER_IDS["bert"])
            model.to(device)
            model.eval()
            _model_cache[cache_key] = (tokenizer, model)

        tokenizer, model = _model_cache[cache_key]
        import torch

        embeddings = []
        model.eval()
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]
                encoded = tokenizer(batch_texts, max_length=64, truncation=True,
                                   padding=True, return_tensors="pt")
                input_ids = encoded["input_ids"].to(device)
                attention_mask = encoded["attention_mask"].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                # Take CLS token (first token of last hidden state)
                cls_tokens = outputs.last_hidden_state[:, 0, :]  # shape: (batch_size, 768)

                # L2 normalize
                cls_normalized = torch.nn.functional.normalize(cls_tokens, p=2, dim=1)
                embeddings.append(cls_normalized.cpu().numpy())

        embeddings_array = np.vstack(embeddings).astype(np.float32)
        return embeddings_array


def embed_texts_tokenwise(texts: list[str], encoder_name: str, device: str,
                          max_length: int = 64, batch_size: int = 32):
    """
    Token-level embeddings for the attention-pooling method (`run_intermediate_attn`).

    Supports BOTH encoders. `all-MiniLM-L6-v2` is distributed as a
    sentence-transformers checkpoint, but the underlying network is an ordinary
    BERT-family transformer, so `AutoModel.from_pretrained` loads it directly and
    exposes `last_hidden_state`. Routing both encoders through AutoModel here is
    what closes the `intermediate_attn / minilm` gap — the first run reported NaN
    for that cell purely because this function used to refuse anything but bert,
    and a method that only ever runs on one encoder cannot support a robustness
    claim (we already know early-vs-intermediate flips between encoders).

    NOTE on a deliberate asymmetry: the POOLED path (`embed_texts`) keeps using
    sentence-transformers for minilm, with its mean-pooling + normalisation,
    because that is what produced the already-reported pooled results. The
    token-level path here is raw `last_hidden_state` for both encoders. So minilm
    has two slightly different representations depending on the path. That is
    intentional — changing the pooled path would invalidate existing numbers.

    Returns (embeddings, attention_mask):
        embeddings: np.ndarray (N, T, D) float32, last_hidden_state (no CLS pooling)
        attention_mask: np.ndarray (N, T) int, 1 for real tokens, 0 for padding
    T = max_length for every row (padded/truncated), so batches concatenate cleanly.
    """
    from common import ENCODER_IDS

    if encoder_name not in ENCODER_IDS:
        raise ValueError(
            f"unknown encoder {encoder_name!r}, expected one of {sorted(ENCODER_IDS)}"
        )

    import torch
    from transformers import AutoTokenizer, AutoModel

    # Separate cache namespace from the pooled loader: for minilm the pooled
    # path stores a SentenceTransformer under ("minilm", device), and handing
    # that object to this function would break it.
    cache_key = ("tokenwise", encoder_name, device)
    if cache_key not in _model_cache:
        model_id = ENCODER_IDS[encoder_name]
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModel.from_pretrained(model_id)
        model.to(device)
        model.eval()
        _model_cache[cache_key] = (tokenizer, model)
    tokenizer, model = _model_cache[cache_key]

    all_hidden = []
    all_mask = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded = tokenizer(
                batch_texts, max_length=max_length, truncation=True,
                padding="max_length", return_tensors="pt",
            )
            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_hidden.append(outputs.last_hidden_state.cpu().numpy())
            all_mask.append(attention_mask.cpu().numpy())

    hidden = np.concatenate(all_hidden, axis=0).astype(np.float32)
    mask = np.concatenate(all_mask, axis=0).astype(np.int64)
    return hidden, mask


def get_token_embeddings(splits: dict, encoder_name: str, device: str,
                         cache_dir: str = "/content/emb_cache_tok",
                         max_length: int = 64) -> dict:
    """
    Token-level embedding cache for `run_intermediate_attn`.

    Mirrors `get_embeddings`'s caching discipline (per split, keyed on row
    count + SHA1 of the uid list) but writes to a SEPARATE cache directory
    and uses a different filename prefix ("tok_" vs the pooled cache's bare
    encoder name), so it never collides with or silently reuses a pooled
    cache file. Does NOT touch or alter `get_embeddings`.

    Returns {"train"/"val"/"test": (embeddings (N, T, D) float32, attention_mask (N, T) int)}.
    Cached to disk as float16 to keep the cache directory small; loaded back
    as float32 for use, per the contract's float32 requirement for in-memory Emb.
    """
    os.makedirs(cache_dir, exist_ok=True)
    out = {}

    for split_name in ["train", "val", "test"]:
        rows = splits[split_name]
        n = len(rows)
        uids = [row["uid"] for row in rows]
        uid_hash = hashlib.sha1(",".join(str(u) for u in uids).encode()).hexdigest()

        emb_file = os.path.join(cache_dir, f"tok_{encoder_name}_{split_name}_{n}_{uid_hash}_emb.npy")
        mask_file = os.path.join(cache_dir, f"tok_{encoder_name}_{split_name}_{n}_{uid_hash}_mask.npy")

        if os.path.exists(emb_file) and os.path.exists(mask_file):
            hidden = np.load(emb_file).astype(np.float32)
            mask = np.load(mask_file)
        else:
            texts = [row["text"] for row in rows]
            hidden, mask = embed_texts_tokenwise(texts, encoder_name, device, max_length=max_length)
            np.save(emb_file, hidden.astype(np.float16))  # cache compactly on disk
            np.save(mask_file, mask)

        assert hidden.shape[0] == n, (
            f"Token embedding row count {hidden.shape[0]} != split row count {n} for {split_name}"
        )
        out[split_name] = (hidden.astype(np.float32), mask)

    return out


def get_embeddings(splits: dict, encoder_name: str, device: str,
                  cache_dir: str = "/content/emb_cache") -> dict:
    """
    Get embeddings for all three splits (train, val, test).

    Caches embeddings to disk with a key that includes encoder_name, split name, row count,
    and a SHA1 hash of the uid list. This ensures that different data variants (full vs filtered)
    produce different cache files and never silently reuse a cache from a different variant.

    Args:
        splits: {"train": list[Row], "val": list[Row], "test": list[Row]}
                where Row = dict with at least "uid" and "text" fields
        encoder_name: "minilm" or "bert"
        device: "cpu" or "cuda"
        cache_dir: directory for embedding cache files (default "/content/emb_cache")

    Returns:
        {"train": np.ndarray, "val": np.ndarray, "test": np.ndarray}
        All arrays are float32. Train/val/test shapes are (N_train, D), (N_val, D), (N_test, D).
    """
    os.makedirs(cache_dir, exist_ok=True)

    embeddings = {}

    for split_name in ["train", "val", "test"]:
        rows = splits[split_name]
        n = len(rows)

        # Build cache key: encoder_name, split name, row count, uid hash
        # The uid hash ensures different data variants get different cache files
        uids = [row["uid"] for row in rows]
        uid_hash = hashlib.sha1(",".join(str(u) for u in uids).encode()).hexdigest()

        cache_file = os.path.join(cache_dir, f"{encoder_name}_{split_name}_{n}_{uid_hash}.npy")

        if os.path.exists(cache_file):
            # Load from cache
            emb = np.load(cache_file)
        else:
            # Encode texts
            texts = [row["text"] for row in rows]
            emb = embed_texts(texts, encoder_name, device)

            # Save to cache
            np.save(cache_file, emb)

        # Verify row count matches split size before returning
        assert emb.shape[0] == n, \
            f"Embedding row count {emb.shape[0]} != split row count {n} for {split_name}"

        embeddings[split_name] = emb.astype(np.float32)

    return embeddings

In [ ]:
%%writefile late.py
"""
Module 3: LATE (decision-level) fusion for the fusion benchmark.

Two independent unimodal classifiers trained separately, combined only at the decision level
via sklearn LogisticRegression over concatenated softmax probabilities.

Extended (without changing `run_late`'s observable behaviour) with:
  - a class-weighted loss option, gated by common.CLASS_WEIGHTING,
  - a parametrised `_run_late_core` used both by `run_late` (fixed defaults,
    matching the original contract hyperparameters exactly) and by
    sweep.py (varies lr / text-branch hidden size / dropout),
  - a `run_late_logits` variant that also returns val/test combiner scores
    for the post-hoc class-bias correction step.
"""

import numpy as np
import copy
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

import common as _common
from common import set_seed, y_of, emotion_features, LABELS, class_weights
# CLASS_WEIGHTING is read live off `_common.CLASS_WEIGHTING` at call time (see
# early.py's comment for why: a `from ... import CLASS_WEIGHTING`
# would freeze the flag's value at import time and ignore later toggles).


def run_late(
    splits: dict,
    emb: dict,
    seed: int,
    device: str,
    encoder_name: str
) -> tuple[np.ndarray, np.ndarray]:
    """
    LATE (decision-level) fusion: two independent branches combined at the final layer.

    Returns (y_true_test, y_pred_test), both int arrays of shape (N_test,),
    values in 0..2 indexing LABELS.
    """
    result = _run_late_core(splits, emb, seed, device, encoder_name)
    return result["y_test"], result["test_pred"]


def run_late_logits(splits, emb, seed, device, encoder_name):
    """
    Same computation as `run_late`, also returning combiner scores (the
    LogisticRegression combiner's `decision_function` output, i.e. its
    pre-softmax scores) on val and test, for the post-hoc class-bias
    correction step. No retraining involved.

    Returns (y_val, val_logits, y_test, test_logits).
    """
    result = _run_late_core(splits, emb, seed, device, encoder_name)
    return result["y_val"], result["val_logits"], result["y_test"], result["test_logits"]


def _run_late_core(
    splits: dict,
    emb: dict,
    seed: int,
    device: str,
    encoder_name: str,
    text_hidden: int = 128,
    text_dropout: float = 0.2,
    lr: float = 1e-3,
) -> dict:
    """
    Shared core used by `run_late` (defaults = original contract hyperparameters,
    so `run_late`'s behaviour is unchanged) and by the sweep in
    sweep.py (which varies text_hidden / text_dropout / lr).

    `text_hidden` / `text_dropout` / `lr` apply ONLY to the text branch — the
    emotion branch (6 -> 32 -> 3) is fixed per the contract, since the sweep
    spec calls "hidden" specifically "the text-branch hidden size" for late.

    Returns a dict with y_val, val_pred, val_logits, y_test, test_pred,
    test_logits (logits = combiner decision_function scores, i.e. pre-argmax).
    """
    set_seed(seed)

    # Lazy torch import
    import torch
    import torch.nn as nn

    # Extract data
    y_train = y_of(splits["train"])
    y_val = y_of(splits["val"])
    y_test = y_of(splits["test"])

    X_emotion_train = emotion_features(splits["train"], "train", seed).astype(np.float32)  # (N_tr, 6)
    X_emotion_val = emotion_features(splits["val"], "val", seed).astype(np.float32)      # (N_val, 6)
    X_emotion_test = emotion_features(splits["test"], "test", seed).astype(np.float32)    # (N_test, 6)

    X_text_train = emb["train"].astype(np.float32)  # (N_tr, D)
    X_text_val = emb["val"].astype(np.float32)      # (N_val, D)
    X_text_test = emb["test"].astype(np.float32)    # (N_test, D)

    D = X_text_train.shape[1]

    # Define emotion branch architecture (fixed shape per contract)
    class EmotionBranch(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(6, 32)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(32, 3)

        def forward(self, x):
            x = self.relu(self.fc1(x))
            x = self.fc2(x)
            return x

    # Define text branch architecture (hidden/dropout parametrised for the sweep)
    class TextBranch(nn.Module):
        def __init__(self, input_dim, hidden, dropout):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, hidden)
            self.relu = nn.ReLU()
            self.dropout = nn.Dropout(dropout)
            self.fc2 = nn.Linear(hidden, 3)

        def forward(self, x):
            x = self.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.fc2(x)
            return x

    class_w = class_weights(y_train) if _common.CLASS_WEIGHTING else None

    # Train emotion branch
    emotion_branch = _train_branch(
        X_emotion_train, y_train, X_emotion_val, y_val,
        EmotionBranch(), device, batch_size=64, max_epochs=40, patience=5,
        lr=1e-3, class_w=class_w,
    )
    emotion_branch = emotion_branch.to(device)
    emotion_branch.eval()

    # Train text branch
    text_branch = _train_branch(
        X_text_train, y_train, X_text_val, y_val,
        TextBranch(D, text_hidden, text_dropout), device, batch_size=64, max_epochs=40, patience=5,
        lr=lr, class_w=class_w,
    )
    text_branch = text_branch.to(device)
    text_branch.eval()

    # Get probabilities on val set (for combiner training)
    with torch.no_grad():
        emotion_val_logits = emotion_branch(torch.tensor(X_emotion_val, device=device))
        emotion_val_probs = torch.softmax(emotion_val_logits, dim=1).cpu().numpy()

        text_val_logits = text_branch(torch.tensor(X_text_val, device=device))
        text_val_probs = torch.softmax(text_val_logits, dim=1).cpu().numpy()

    # Concatenate val probabilities: [emotion_probs (3), text_probs (3)] -> (N_val, 6)
    X_combiner_val = np.hstack([emotion_val_probs, text_val_probs])

    # Fit combiner on val probabilities. class_weight="balanced" honours the
    # same all-or-nothing CLASS_WEIGHTING switch as the branch losses above.
    combiner = LogisticRegression(
        max_iter=1000, class_weight=("balanced" if _common.CLASS_WEIGHTING else None)
    )
    combiner.fit(X_combiner_val, y_val)

    # Late fusion has no single model: it is two independent branches plus an
    # sklearn combiner, so all three are offered together. No-op unless the
    # driver installed a sink.
    _common.offer_checkpoint(
        f"late_{encoder_name}_{seed}",
        {
            "emotion_branch": emotion_branch.state_dict(),
            "text_branch": text_branch.state_dict(),
            "combiner_coef": combiner.coef_,
            "combiner_intercept": combiner.intercept_,
        },
        f1_score(y_val, combiner.predict(X_combiner_val), average="macro", zero_division=0),
        meta={"method": "late", "encoder": encoder_name, "seed": seed,
              "regime": _common.EMOTION_REGIME,
              "class_weighting": _common.CLASS_WEIGHTING},
    )

    # Get probabilities on test set (for combiner prediction)
    with torch.no_grad():
        emotion_test_logits = emotion_branch(torch.tensor(X_emotion_test, device=device))
        emotion_test_probs = torch.softmax(emotion_test_logits, dim=1).cpu().numpy()

        text_test_logits = text_branch(torch.tensor(X_text_test, device=device))
        text_test_probs = torch.softmax(text_test_logits, dim=1).cpu().numpy()

    X_combiner_test = np.hstack([emotion_test_probs, text_test_probs])

    val_logits = combiner.decision_function(X_combiner_val)
    test_logits = combiner.decision_function(X_combiner_test)
    val_pred = combiner.predict(X_combiner_val).astype(np.int64)
    test_pred = combiner.predict(X_combiner_test).astype(np.int64)

    return {
        "y_val": y_val.astype(np.int64),
        "val_pred": val_pred,
        "val_logits": val_logits,
        "y_test": y_test.astype(np.int64),
        "test_pred": test_pred,
        "test_logits": test_logits,
    }


def _train_branch(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    model,
    device: str,
    batch_size: int = 64,
    max_epochs: int = 40,
    patience: int = 5,
    lr: float = 1e-3,
    class_w: np.ndarray | None = None,
):
    """
    Train a single branch model with early stopping on val macro-F1.

    Returns the model with best state dict restored.
    """
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader

    model = model.to(device)

    # Create data loaders
    train_dataset = TensorDataset(
        torch.tensor(X_train, device=device),
        torch.tensor(y_train, device=device, dtype=torch.long)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    val_dataset = TensorDataset(
        torch.tensor(X_val, device=device),
        torch.tensor(y_val, device=device, dtype=torch.long)
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Loss and optimizer. class_w is None unless CLASS_WEIGHTING is on, in
    # which case it is the same inverse-frequency weight vector for both
    # branches (computed once on train labels, in the caller).
    if class_w is not None:
        weight_tensor = torch.tensor(class_w, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Early stopping state
    best_val_f1 = -np.inf
    best_state_dict = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        # Training
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_preds = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                logits = model(X_batch)
                preds = torch.argmax(logits, dim=1)
                val_preds.append(preds.cpu().numpy())

        val_preds_all = np.concatenate(val_preds)
        val_f1 = f1_score(y_val, val_preds_all, average="macro", zero_division=0)

        # Early stopping check
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    # Restore best state
    model.load_state_dict(best_state_dict)
    model.eval()

    return model

In [ ]:
%%writefile inter.py
"""
Module 4: INTERMEDIATE (feature-level) fusion for the fusion benchmark.

Extended (without changing `run_intermediate`'s observable behaviour) with:
  - a class-weighted loss option, gated by common.CLASS_WEIGHTING,
  - a parametrised `_run_intermediate_core` used both by `run_intermediate`
    (fixed defaults matching the original contract exactly) and by
    sweep.py (varies lr / fc1 hidden size / dropout),
  - a `run_intermediate_logits` variant returning val/test logits for the
    post-hoc class-bias correction step.
"""

import numpy as np
import copy
from sklearn.metrics import f1_score
import common as _common
from common import set_seed, y_of, emotion_features, class_weights
# CLASS_WEIGHTING read live off `_common.CLASS_WEIGHTING` — see early.py.


class IntermediateFusionModel:
    """Placeholder for type hints only; the real nn.Module is built lazily
    inside _run_intermediate_core so this module stays torch-free at import
    time."""
    pass


def run_intermediate(splits, emb, seed, device, encoder_name):
    """
    INTERMEDIATE fusion: concatenate learned emotion embedding with frozen text embedding
    as features, then jointly transform through MLP.

    Args:
        splits: dict with "train", "val", "test" keys, each containing list[Row]
        emb: dict with "train", "val", "test" keys, each containing np.ndarray embeddings
        seed: int, random seed
        device: str, "cpu" or "cuda"
        encoder_name: str, encoder identifier (not used here)

    Returns:
        tuple[np.ndarray, np.ndarray]: (y_true_test, y_pred_test) as int arrays
    """
    result = _run_intermediate_core(splits, emb, seed, device, encoder_name)
    return result["y_test"], result["test_pred"]


def run_intermediate_logits(splits, emb, seed, device, encoder_name):
    """
    Same computation as `run_intermediate`, also returning val/test logits
    for the post-hoc class-bias correction step. No retraining.

    Returns (y_val, val_logits, y_test, test_logits).
    """
    result = _run_intermediate_core(splits, emb, seed, device, encoder_name)
    return result["y_val"], result["val_logits"], result["y_test"], result["test_logits"]


def _run_intermediate_core(
    splits, emb, seed, device, encoder_name,
    hidden: int = 256, dropout: float = 0.3, lr: float = 1e-3,
) -> dict:
    """
    Shared core used by `run_intermediate` (defaults = original contract
    hyperparameters) and by sweep.py (varies hidden/dropout/lr).

    `hidden` is the fc1 output size (32 + text_dim -> hidden); fc2 (-> 128)
    and the emotion embedding (6 -> 32) are fixed, matching the contract
    shape and matching how "hidden" is defined for `late`'s text branch.

    Returns a dict with y_val, val_pred, val_logits, y_test, test_pred, test_logits.
    """
    set_seed(seed)

    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    from torch.optim import Adam

    emotion_feat_train = emotion_features(splits["train"], "train", seed)
    emotion_feat_val = emotion_features(splits["val"], "val", seed)
    emotion_feat_test = emotion_features(splits["test"], "test", seed)

    text_emb_train = emb["train"]
    text_emb_val = emb["val"]
    text_emb_test = emb["test"]

    y_train = y_of(splits["train"])
    y_val = y_of(splits["val"])
    y_test = y_of(splits["test"])

    D = text_emb_train.shape[1]

    class _IntermediateFusionModel(nn.Module):
        def __init__(self, text_dim, hidden):
            super().__init__()
            self.emotion_embedding = nn.Linear(6, 32, bias=False)
            self.fc1 = nn.Linear(32 + text_dim, hidden)
            self.relu1 = nn.ReLU()
            self.dropout1 = nn.Dropout(dropout)
            self.fc2 = nn.Linear(hidden, 128)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(128, 3)

        def forward(self, emo_idx, text_emb):
            emotion_emb = self.emotion_embedding(emo_idx)
            combined = torch.cat([emotion_emb, text_emb], dim=1)
            x = self.fc1(combined)
            x = self.relu1(x)
            x = self.dropout1(x)
            x = self.fc2(x)
            x = self.relu2(x)
            logits = self.fc3(x)
            return logits

    model = _IntermediateFusionModel(D, hidden).to(device)

    emotion_feat_train_t = torch.FloatTensor(emotion_feat_train).to(device)
    emotion_feat_val_t = torch.FloatTensor(emotion_feat_val).to(device)
    emotion_feat_test_t = torch.FloatTensor(emotion_feat_test).to(device)

    text_emb_train_t = torch.FloatTensor(text_emb_train).to(device)
    text_emb_val_t = torch.FloatTensor(text_emb_val).to(device)
    text_emb_test_t = torch.FloatTensor(text_emb_test).to(device)

    y_train_t = torch.LongTensor(y_train).to(device)
    y_val_t = torch.LongTensor(y_val).to(device)

    train_dataset = TensorDataset(emotion_feat_train_t, text_emb_train_t, y_train_t)
    val_dataset = TensorDataset(emotion_feat_val_t, text_emb_val_t, y_val_t)
    test_dataset = TensorDataset(emotion_feat_test_t, text_emb_test_t)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=generator)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    if _common.CLASS_WEIGHTING:
        w = torch.tensor(class_weights(y_train), dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=w)
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_f1 = -1
    best_state_dict = None
    best_val_logits = None
    patience = 8
    patience_counter = 0

    for epoch in range(60):
        model.train()
        for emotion_batch, text_emb_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(emotion_batch, text_emb_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logit_chunks = []
            for emotion_batch, text_emb_batch, _ in val_loader:
                logits = model(emotion_batch, text_emb_batch)
                val_logit_chunks.append(logits.cpu().numpy())
            val_logits_epoch = np.concatenate(val_logit_chunks, axis=0)
            y_val_pred = np.argmax(val_logits_epoch, axis=1)

        val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state_dict = copy.deepcopy(model.state_dict())
            best_val_logits = val_logits_epoch
            patience_counter = 0
            print(f"Epoch {epoch+1}: F1={val_f1:.4f}")

        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Hand the trained model to the driver if it asked for one (no-op
    # during sweeps, which train hundreds of throwaway models).
    _common.offer_checkpoint(
        f"intermediate_{encoder_name}_{seed}", best_state_dict, best_val_f1,
        meta={"method": "intermediate", "encoder": encoder_name, "seed": seed,
              "regime": _common.EMOTION_REGIME,
              "class_weighting": _common.CLASS_WEIGHTING},
    )
    model.load_state_dict(best_state_dict)
    model.eval()

    with torch.no_grad():
        test_logit_chunks = []
        for emotion_batch, text_emb_batch in test_loader:
            logits = model(emotion_batch, text_emb_batch)
            test_logit_chunks.append(logits.cpu().numpy())
        test_logits = np.concatenate(test_logit_chunks, axis=0)
        y_test_pred = np.argmax(test_logits, axis=1)

    return {
        "y_val": y_val.astype(np.int64),
        "val_pred": np.argmax(best_val_logits, axis=1).astype(np.int64),
        "val_logits": best_val_logits,
        "y_test": y_test.astype(np.int64),
        "test_pred": y_test_pred.astype(np.int64),
        "test_logits": test_logits,
    }

In [ ]:
%%writefile film.py
"""
film.py — FiLM-conditioned intermediate fusion.

Deliverable #5 of the extension: a SEPARATE method row from `run_intermediate`
(inter.py), which that module is left untouched by.

Rationale (from the benchmark spec): in `run_intermediate`, the emotion
embedding is 32 dims concatenated against a 768-dim (bert) or 384-dim
(minilm) text embedding — only ~4-8% of the input to the first Linear layer.
A plain MLP can learn to all but ignore that slice. FiLM (Feature-wise Linear
Modulation) instead uses the emotion embedding to produce a per-dimension
scale (`gamma`) and shift (`beta`) that multiplicatively/additively modulate
the FULL text embedding, so emotion has leverage over every text dimension
rather than competing for capacity as 4% of concatenated width.

Architecture:
  emotion embedding (nn.Linear(6, 32, bias=False))
    -> Linear(32, D) = gamma_head  -> gamma (N, D)
    -> Linear(32, D) = beta_head   -> beta  (N, D)
  h = gamma * text_emb + beta                      # FiLM modulation
  h -> Linear(D, hidden) -> ReLU -> Dropout -> Linear(hidden, 128) -> ReLU -> Linear(128, 3)
  (same MLP trunk shape as run_intermediate, after the fusion point)

Same training loop, early stopping, class-weighting switch, and equal sweep
budget as `run_intermediate` (see sweep.py).
"""

import numpy as np
import copy
from sklearn.metrics import f1_score
import common as _common
from common import set_seed, y_of, emotion_features, class_weights
# CLASS_WEIGHTING read live off `_common.CLASS_WEIGHTING` — see early.py.


def run_intermediate_film(splits, emb, seed, device, encoder_name):
    """
    FiLM-conditioned intermediate fusion. Registered as a SEPARATE method row
    from `run_intermediate` — does not modify or call into that module.

    Returns (y_true_test, y_pred_test), both int arrays of shape (N_test,),
    values in 0..2 indexing LABELS.
    """
    result = _run_film_core(splits, emb, seed, device, encoder_name)
    return result["y_test"], result["test_pred"]


def run_intermediate_film_logits(splits, emb, seed, device, encoder_name):
    """Same computation as `run_intermediate_film`, also returning val/test
    logits for the post-hoc class-bias correction step."""
    result = _run_film_core(splits, emb, seed, device, encoder_name)
    return result["y_val"], result["val_logits"], result["y_test"], result["test_logits"]


def _run_film_core(
    splits, emb, seed, device, encoder_name,
    hidden: int = 256, dropout: float = 0.3, lr: float = 1e-3,
) -> dict:
    """
    Core FiLM training loop. Hyperparameters default to the same values as
    `run_intermediate`'s defaults so the two are compared on equal footing;
    sweep.py varies hidden/dropout/lr with an identical grid.

    Returns a dict with y_val, val_pred, val_logits, y_test, test_pred, test_logits.
    """
    set_seed(seed)

    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    from torch.optim import Adam

    emotion_feat_train = emotion_features(splits["train"], "train", seed)
    emotion_feat_val = emotion_features(splits["val"], "val", seed)
    emotion_feat_test = emotion_features(splits["test"], "test", seed)

    # --- FIX 2: mean-centre the text features -------------------------------
    # BERT CLS embeddings are strongly anisotropic: after L2 normalisation the
    # vectors all sit in a narrow cone (cosine similarity ~0.9 between random
    # sentences), so most of each vector is a large shared component and only a
    # small residual actually distinguishes utterances. FiLM is MULTIPLICATIVE,
    # so `gamma * text` amplifies that shared component and the product becomes
    # almost a pure function of the emotion — the model collapses onto the
    # emotion-only solution (val macro-F1 0.4141, which is exactly the
    # emotion-only score) and never recovers. Removing the mean leaves the
    # discriminative residual for gamma to modulate.
    # Statistics come from TRAIN ONLY — computing them over val/test would leak.
    text_mean = emb["train"].mean(axis=0, keepdims=True)
    text_emb_train = emb["train"] - text_mean
    text_emb_val = emb["val"] - text_mean
    text_emb_test = emb["test"] - text_mean

    y_train = y_of(splits["train"])
    y_val = y_of(splits["val"])
    y_test = y_of(splits["test"])

    D = text_emb_train.shape[1]

    class _FiLMFusionModel(nn.Module):
        def __init__(self, text_dim, hidden):
            super().__init__()
            self.emotion_embedding = nn.Linear(6, 32, bias=False)
            self.gamma_head = nn.Linear(32, text_dim)
            self.beta_head = nn.Linear(32, text_dim)
            self.fc1 = nn.Linear(text_dim, hidden)
            self.relu1 = nn.ReLU()
            self.dropout1 = nn.Dropout(dropout)
            self.fc2 = nn.Linear(hidden, 128)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(128, 3)

            # --- FIX 1: initialise FiLM as the identity transform ------------
            # With default nn.Linear init, gamma starts near 0, so
            # `h = gamma * text + beta` is ~0 on the first step: the text
            # signal is annihilated before the trunk ever sees it and there is
            # no gradient path back to it. Zeroing the weights and setting
            # gamma's bias to 1 makes the layer start as exactly `h = text`
            # (plain passthrough, i.e. the same starting point as a text-only
            # model) and lets the network LEARN modulation away from identity.
            # This is the standard FiLM initialisation.
            nn.init.zeros_(self.gamma_head.weight)
            nn.init.ones_(self.gamma_head.bias)
            nn.init.zeros_(self.beta_head.weight)
            nn.init.zeros_(self.beta_head.bias)

        def forward(self, emo_idx, text_emb):
            emotion_emb = self.emotion_embedding(emo_idx)      # (N, 32)
            gamma = self.gamma_head(emotion_emb)                # (N, D)
            beta = self.beta_head(emotion_emb)                  # (N, D)
            h = gamma * text_emb + beta                          # FiLM modulation, full width
            x = self.fc1(h)
            x = self.relu1(x)
            x = self.dropout1(x)
            x = self.fc2(x)
            x = self.relu2(x)
            logits = self.fc3(x)
            return logits

    model = _FiLMFusionModel(D, hidden).to(device)

    emotion_feat_train_t = torch.FloatTensor(emotion_feat_train).to(device)
    emotion_feat_val_t = torch.FloatTensor(emotion_feat_val).to(device)
    emotion_feat_test_t = torch.FloatTensor(emotion_feat_test).to(device)

    text_emb_train_t = torch.FloatTensor(text_emb_train).to(device)
    text_emb_val_t = torch.FloatTensor(text_emb_val).to(device)
    text_emb_test_t = torch.FloatTensor(text_emb_test).to(device)

    y_train_t = torch.LongTensor(y_train).to(device)
    y_val_t = torch.LongTensor(y_val).to(device)

    train_dataset = TensorDataset(emotion_feat_train_t, text_emb_train_t, y_train_t)
    val_dataset = TensorDataset(emotion_feat_val_t, text_emb_val_t, y_val_t)
    test_dataset = TensorDataset(emotion_feat_test_t, text_emb_test_t)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=generator)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    if _common.CLASS_WEIGHTING:
        w = torch.tensor(class_weights(y_train), dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=w)
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_f1 = -1
    best_state_dict = None
    best_val_logits = None
    patience = 8
    patience_counter = 0

    for epoch in range(60):
        model.train()
        for emotion_batch, text_emb_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(emotion_batch, text_emb_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logit_chunks = []
            for emotion_batch, text_emb_batch, _ in val_loader:
                logits = model(emotion_batch, text_emb_batch)
                val_logit_chunks.append(logits.cpu().numpy())
            val_logits_epoch = np.concatenate(val_logit_chunks, axis=0)
            y_val_pred = np.argmax(val_logits_epoch, axis=1)

        val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state_dict = copy.deepcopy(model.state_dict())
            best_val_logits = val_logits_epoch
            patience_counter = 0
            print(f"[film] Epoch {epoch+1}: F1={val_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Hand the trained model to the driver if it asked for one (no-op
    # during sweeps, which train hundreds of throwaway models).
    _common.offer_checkpoint(
        f"intermediate_film_{encoder_name}_{seed}", best_state_dict, best_val_f1,
        meta={"method": "intermediate_film", "encoder": encoder_name, "seed": seed,
              "regime": _common.EMOTION_REGIME,
              "class_weighting": _common.CLASS_WEIGHTING},
    )
    model.load_state_dict(best_state_dict)
    model.eval()

    with torch.no_grad():
        test_logit_chunks = []
        for emotion_batch, text_emb_batch in test_loader:
            logits = model(emotion_batch, text_emb_batch)
            test_logit_chunks.append(logits.cpu().numpy())
        test_logits = np.concatenate(test_logit_chunks, axis=0)
        y_test_pred = np.argmax(test_logits, axis=1)

    return {
        "y_val": y_val.astype(np.int64),
        "val_pred": np.argmax(best_val_logits, axis=1).astype(np.int64),
        "val_logits": best_val_logits,
        "y_test": y_test.astype(np.int64),
        "test_pred": y_test_pred.astype(np.int64),
        "test_logits": test_logits,
    }

In [ ]:
%%writefile attn.py
"""
attn.py — emotion-conditioned attention pooling over token-level
text features.

Deliverable #6 of the extension. Separate method row from both
`run_intermediate` and `run_intermediate_film`.

Where `run_intermediate` fuses a single pooled (CLS) text vector with the
emotion embedding, this method lets the emotion embedding act as an
attention QUERY over the text encoder's TOKEN-level hidden states — so
emotion can pick out which words in the utterance it is most relevant to,
rather than being concatenated onto one fixed pooled summary.

Requires `encoders.get_token_embeddings`, which returns
per-token BERT hidden states (bert only — see that module's docstring for
why minilm is not supported here).

Architecture:
  emotion embedding (nn.Linear(6, 32, bias=False)) -> Linear(32, D) = query projection
    -> query (N, 1, D)
  token states (N, T, D) = keys and values (already produced by the encoder)
  nn.MultiheadAttention(embed_dim=D, num_heads=4, batch_first=True),
    key_padding_mask = ~attention_mask.bool()   (True = ignore, per torch's convention)
  attended output (N, 1, D) -> squeeze -> (N, D)
    -> same MLP trunk as run_intermediate: Linear(D, hidden) -> ReLU -> Dropout
       -> Linear(hidden, 128) -> ReLU -> Linear(128, 3)

Same training loop, early stopping, class-weighting switch, and equal sweep
budget as `run_intermediate`.
"""

import numpy as np
import copy
from sklearn.metrics import f1_score
import common as _common
from common import set_seed, y_of, emotion_features, class_weights
# CLASS_WEIGHTING read live off `_common.CLASS_WEIGHTING` — see early.py.


def run_intermediate_attn(splits, emb, seed, device, encoder_name):
    """
    Emotion-conditioned attention pooling over token-level text features.

    NOTE on the uniform signature: like every other `run_*` method this
    accepts `emb` (the POOLED embeddings dict) for signature uniformity, but
    this method ignores it and instead calls
    `encoders.get_token_embeddings` itself to obtain per-token
    features — the pooled `emb` cannot be used for attention over tokens.
    Only supported for encoder_name == "bert" (see encoders.py);
    calling it with "minilm" raises ValueError from that function.

    Returns (y_true_test, y_pred_test), both int arrays of shape (N_test,),
    values in 0..2 indexing LABELS.
    """
    result = _run_attn_core(splits, emb, seed, device, encoder_name)
    return result["y_test"], result["test_pred"]


def run_intermediate_attn_logits(splits, emb, seed, device, encoder_name):
    """Same computation as `run_intermediate_attn`, also returning val/test
    logits for the post-hoc class-bias correction step."""
    result = _run_attn_core(splits, emb, seed, device, encoder_name)
    return result["y_val"], result["val_logits"], result["y_test"], result["test_logits"]


def _run_attn_core(
    splits, emb, seed, device, encoder_name,
    hidden: int = 256, dropout: float = 0.3, lr: float = 1e-3,
    cache_dir: str | None = None,
) -> dict:
    """
    Core training loop for emotion-conditioned attention pooling.

    Returns a dict with y_val, val_pred, val_logits, y_test, test_pred, test_logits.
    """
    set_seed(seed)

    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    from torch.optim import Adam
    from encoders import get_token_embeddings

    # Read the cache location live off common so the driver can
    # point it at Drive; otherwise the ~2 GB token cache is rebuilt every session.
    tok = get_token_embeddings(
        splits, encoder_name, device,
        cache_dir=_common.TOKEN_CACHE_DIR if cache_dir is None else cache_dir,
    )
    tok_train, mask_train = tok["train"]
    tok_val, mask_val = tok["val"]
    tok_test, mask_test = tok["test"]

    D = tok_train.shape[2]

    emotion_feat_train = emotion_features(splits["train"], "train", seed)
    emotion_feat_val = emotion_features(splits["val"], "val", seed)
    emotion_feat_test = emotion_features(splits["test"], "test", seed)

    y_train = y_of(splits["train"])
    y_val = y_of(splits["val"])
    y_test = y_of(splits["test"])

    class _AttnFusionModel(nn.Module):
        def __init__(self, text_dim, hidden, num_heads=4):
            super().__init__()
            self.emotion_embedding = nn.Linear(6, 32, bias=False)
            self.query_proj = nn.Linear(32, text_dim)
            self.attn = nn.MultiheadAttention(
                embed_dim=text_dim, num_heads=num_heads, batch_first=True
            )
            self.fc1 = nn.Linear(text_dim, hidden)
            self.relu1 = nn.ReLU()
            self.dropout1 = nn.Dropout(dropout)
            self.fc2 = nn.Linear(hidden, 128)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(128, 3)

        def forward(self, emo_idx, token_states, attn_mask):
            # emo_idx: (N,); token_states: (N, T, D); attn_mask: (N, T) 1=real, 0=pad
            emotion_emb = self.emotion_embedding(emo_idx)          # (N, 32)
            query = self.query_proj(emotion_emb).unsqueeze(1)      # (N, 1, D)
            key_padding_mask = attn_mask == 0                       # True = ignore this position
            attended, _ = self.attn(
                query, token_states, token_states, key_padding_mask=key_padding_mask
            )
            h = attended.squeeze(1)                                 # (N, D)
            x = self.fc1(h)
            x = self.relu1(x)
            x = self.dropout1(x)
            x = self.fc2(x)
            x = self.relu2(x)
            logits = self.fc3(x)
            return logits

    model = _AttnFusionModel(D, hidden).to(device)

    emotion_feat_train_t = torch.FloatTensor(emotion_feat_train).to(device)
    emotion_feat_val_t = torch.FloatTensor(emotion_feat_val).to(device)
    emotion_feat_test_t = torch.FloatTensor(emotion_feat_test).to(device)

    tok_train_t = torch.FloatTensor(tok_train).to(device)
    tok_val_t = torch.FloatTensor(tok_val).to(device)
    tok_test_t = torch.FloatTensor(tok_test).to(device)

    mask_train_t = torch.LongTensor(mask_train).to(device)
    mask_val_t = torch.LongTensor(mask_val).to(device)
    mask_test_t = torch.LongTensor(mask_test).to(device)

    y_train_t = torch.LongTensor(y_train).to(device)
    y_val_t = torch.LongTensor(y_val).to(device)

    train_dataset = TensorDataset(emotion_feat_train_t, tok_train_t, mask_train_t, y_train_t)
    val_dataset = TensorDataset(emotion_feat_val_t, tok_val_t, mask_val_t, y_val_t)
    test_dataset = TensorDataset(emotion_feat_test_t, tok_test_t, mask_test_t)

    generator = torch.Generator()
    generator.manual_seed(seed)

    # Smaller batch than run_intermediate: (N, T, D) token tensors are much
    # larger than pooled (N, D) ones, and this only needs to run on frozen
    # features, so trading batch size for memory headroom costs nothing here.
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=generator)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    if _common.CLASS_WEIGHTING:
        w = torch.tensor(class_weights(y_train), dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=w)
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_f1 = -1
    best_state_dict = None
    best_val_logits = None
    patience = 8
    patience_counter = 0

    for epoch in range(60):
        model.train()
        for emotion_batch, tok_batch, mask_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(emotion_batch, tok_batch, mask_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logit_chunks = []
            for emotion_batch, tok_batch, mask_batch, _ in val_loader:
                logits = model(emotion_batch, tok_batch, mask_batch)
                val_logit_chunks.append(logits.cpu().numpy())
            val_logits_epoch = np.concatenate(val_logit_chunks, axis=0)
            y_val_pred = np.argmax(val_logits_epoch, axis=1)

        val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state_dict = copy.deepcopy(model.state_dict())
            best_val_logits = val_logits_epoch
            patience_counter = 0
            print(f"[attn] Epoch {epoch+1}: F1={val_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Hand the trained model to the driver if it asked for one (no-op
    # during sweeps, which train hundreds of throwaway models).
    _common.offer_checkpoint(
        f"intermediate_attn_{encoder_name}_{seed}", best_state_dict, best_val_f1,
        meta={"method": "intermediate_attn", "encoder": encoder_name, "seed": seed,
              "regime": _common.EMOTION_REGIME,
              "class_weighting": _common.CLASS_WEIGHTING},
    )
    model.load_state_dict(best_state_dict)
    model.eval()

    with torch.no_grad():
        test_logit_chunks = []
        for emotion_batch, tok_batch, mask_batch in test_loader:
            logits = model(emotion_batch, tok_batch, mask_batch)
            test_logit_chunks.append(logits.cpu().numpy())
        test_logits = np.concatenate(test_logit_chunks, axis=0)
        y_test_pred = np.argmax(test_logits, axis=1)

    return {
        "y_val": y_val.astype(np.int64),
        "val_pred": np.argmax(best_val_logits, axis=1).astype(np.int64),
        "val_logits": best_val_logits,
        "y_test": y_test.astype(np.int64),
        "test_pred": y_test_pred.astype(np.int64),
        "test_logits": test_logits,
    }

In [ ]:
%%writefile early.py
import numpy as np
import copy
from sklearn.metrics import f1_score
import common as _common
from common import set_seed, LABELS, EMOTIONS, ENCODER_IDS, y_of, class_weights
# CLASS_WEIGHTING is read live off `_common.CLASS_WEIGHTING` (not imported by
# name) so that flipping the flag on the `common` module object
# AFTER this module has already been imported still takes effect the next
# time run_early() is called. `from common import CLASS_WEIGHTING`
# would instead freeze whatever value was true at import time.


def run_early(splits, emb, seed, device, encoder_name):
    """
    Early fusion: inject emotion as a special token at the front of text,
    then fine-tune the encoder end-to-end for 3-way classification.

    The defining property of early fusion is that emotion enters at the INPUT,
    before any learned representation is formed, allowing the encoder's attention
    to condition every token on it.

    Args:
        splits: dict with "train", "val", "test" lists of Row dicts
        emb: dict with embeddings (ignored for early fusion)
        seed: random seed
        device: torch device string (e.g., "cuda" or "cpu")
        encoder_name: "minilm" or "bert", key to ENCODER_IDS

    Returns:
        tuple[np.ndarray, np.ndarray]: (y_true_test, y_pred_test),
            both int arrays of shape (N_test,) with values in 0..2 indexing LABELS
    """
    y_true, y_pred, _val_logits, _test_logits = _run_early_or_textft(
        splits, emb, seed, device, encoder_name, use_emotion=True
    )
    return y_true, y_pred


def _run_early_or_textft(splits, emb, seed, device, encoder_name, use_emotion: bool):
    """
    Shared implementation for `run_early` (use_emotion=True) and
    `run_text_only_finetuned` (use_emotion=False, see text_ft.py).

    Identical hyperparameters, training loop, and early stopping in both
    cases. The ONLY difference is whether the emotion token is prepended to
    the input string — this isolates "emotion visible to the encoder" from
    "encoder was fine-tuned", which the `early` vs `text_only` (frozen
    embedding, LogisticRegression) comparison could not do on its own.

    Also returns (val_logits, test_logits) alongside (y_true_test, y_pred_test)
    so the post-hoc class-bias correction (report.tune_class_bias)
    can be applied to this method without retraining.
    """
    set_seed(seed)

    # Lazy imports to avoid requiring torch/transformers at module load time
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    # emb is ignored for early fusion / text-only-finetuned — the point of both
    # is that they build their own representation from the raw string, not
    # from frozen features.

    train_texts = [row["text"] for row in splits["train"]]
    train_emotions = [row["gen_emotion"] for row in splits["train"]]
    train_labels = y_of(splits["train"])

    val_texts = [row["text"] for row in splits["val"]]
    val_emotions = [row["gen_emotion"] for row in splits["val"]]
    val_labels = y_of(splits["val"])

    test_texts = [row["text"] for row in splits["test"]]
    test_emotions = [row["gen_emotion"] for row in splits["test"]]
    test_labels = y_of(splits["test"])

    # Emotion channel as a (N, 6) distribution, honouring the active
    # EMOTION_REGIME. Under "oracle" these are one-hot, so everything below
    # reduces exactly to prepending the true emotion token — the previously
    # reported hard-label numbers stay reproducible.
    train_emo_probs = _common.emotion_features(splits["train"], "train", seed)
    val_emo_probs = _common.emotion_features(splits["val"], "val", seed)
    test_emo_probs = _common.emotion_features(splits["test"], "test", seed)

    if use_emotion:
        # A FIXED placeholder emotion token is prepended to every row, so the
        # tokenisation is identical across rows and the emotion always lands at
        # position 1 (right after [CLS]). The placeholder's EMBEDDING is then
        # overwritten at forward time with the probability-weighted mixture of
        # the six emotion-token embeddings (see `_soft_inputs_embeds`).
        #
        # Why not just prepend the argmax token? Because the SER emits a
        # distribution, and collapsing it to one token throws away its
        # uncertainty — which is exactly the information a combiner needs in
        # order to distrust an unreliable emotion reading. Mixing in embedding
        # space is the input-level equivalent of feeding a soft label.
        placeholder = f"[{EMOTIONS[0].upper()}]"
        train_texts_aug = [f"{placeholder} {t}" for t in train_texts]
        val_texts_aug = [f"{placeholder} {t}" for t in val_texts]
        test_texts_aug = [f"{placeholder} {t}" for t in test_texts]
    else:
        # text_only_finetuned: plain text, no emotion signal anywhere in the input
        train_texts_aug = list(train_texts)
        val_texts_aug = list(val_texts)
        test_texts_aug = list(test_texts)

    # Load tokenizer and model
    model_id = ENCODER_IDS[encoder_name]
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # --- Checkpoint loading guard -----------------------------------------
    # Loading bert-base-uncased for sequence classification prints two kinds
    # of warning; both are EXPECTED here, but the same message would also
    # fire if the encoder body itself silently failed to load, so we check
    # the categories explicitly instead of trusting eyeballed log output:
    #   - UNEXPECTED "cls.predictions.*"/"cls.seq_relationship.*" (or
    #     "pooler.*"): the pretraining MLM/NSP heads in the checkpoint have
    #     no home in AutoModelForSequenceClassification and are discarded —
    #     harmless, we never wanted those heads.
    #   - MISSING "classifier.weight"/"classifier.bias" (or "bert.pooler"/
    #     "pooler"): the 3-way classification head does not exist in a
    #     pretrained MLM checkpoint and is randomly initialised — expected,
    #     because we are about to fine-tune it from scratch.
    # Anything else missing would mean the ENCODER BODY itself failed to
    # load (e.g. a name mismatch / truncated checkpoint) — that must raise,
    # not be silently swallowed as "just the usual warning".
    model, loading_info = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=3, output_loading_info=True
    )
    unexpected_ok = all(
        k.startswith(("cls.", "pooler.")) for k in loading_info["unexpected_keys"]
    )
    assert unexpected_ok, (
        f"unexpected keys outside the known-benign MLM/NSP/pooler heads: "
        f"{loading_info['unexpected_keys']}"
    )
    missing_bad = [
        k for k in loading_info["missing_keys"]
        if not k.startswith(("classifier", "bert.pooler", "pooler"))
    ]
    assert not missing_bad, f"encoder weights missing from checkpoint: {missing_bad}"

    # Add emotion tokens as additional special tokens (added even when
    # use_emotion is False, so the tokenizer/vocab is identical between the
    # two runs and any difference in result is attributable only to whether
    # the token is actually used in the input string, not to a vocab-size
    # side effect).
    emotion_tokens = [f"[{e.upper()}]" for e in EMOTIONS]
    tokenizer.add_special_tokens({"additional_special_tokens": emotion_tokens})

    # CRITICAL: resize_token_embeddings MUST be called AFTER both:
    # (1) tokenizer gains the new tokens, and (2) model is loaded.
    # Calling it in the wrong order produces silent garbage embeddings for new tokens.
    old_vocab_size = model.get_input_embeddings().weight.shape[0]
    model.resize_token_embeddings(len(tokenizer))

    # Initialise the new emotion-token embedding rows from the MEAN of the
    # existing input embeddings, rather than leaving them at whatever
    # `resize_token_embeddings` defaults to (effectively random). With only
    # 4 epochs at lr 2e-5 those rows barely move from their initial value,
    # so a random start would leave them close to noise all the way through
    # training; the mean-of-existing-rows start is a much better prior for
    # "a token that behaves like an ordinary token".
    with torch.no_grad():
        input_embeddings = model.get_input_embeddings()
        mean_embedding = input_embeddings.weight[:old_vocab_size].mean(dim=0)
        input_embeddings.weight[old_vocab_size:] = mean_embedding
        # Some architectures tie input/output embeddings for the LM head;
        # AutoModelForSequenceClassification has no LM head so there is
        # nothing further to tie here, but keep the call in case of a
        # future architecture change.
        model.tie_weights()

    model.to(device)

    # Row indices of the six emotion tokens in the (resized) embedding table.
    emotion_token_ids = torch.tensor(
        tokenizer.convert_tokens_to_ids(emotion_tokens), dtype=torch.long, device=device
    )
    assert (emotion_token_ids >= old_vocab_size).all(), (
        "emotion tokens were not added as NEW vocabulary entries — they must not "
        "collide with pre-existing ids, or the soft mixture would blend unrelated "
        f"wordpieces. ids={emotion_token_ids.tolist()} old_vocab={old_vocab_size}"
    )

    def _soft_inputs_embeds(input_ids, emo_probs):
        """Embed `input_ids`, then replace position 1 with the soft emotion mix.

        `emo_probs` is (B, 6) summing to 1. The mixture is
        `emo_probs @ E[emotion_token_ids]`, i.e. a convex combination of the six
        emotion-token embeddings. With a one-hot input this returns exactly the
        embedding of the true emotion token, so the "oracle" regime is bit-for-bit
        the old hard-token behaviour.

        Gradients flow into the emotion token rows through this mixture, so those
        rows still get trained exactly as they did in the hard-token version.
        """
        emb_layer = model.get_input_embeddings()
        base = emb_layer(input_ids)                       # (B, T, H)
        emotion_matrix = emb_layer.weight[emotion_token_ids]   # (6, H)
        mixed = emo_probs @ emotion_matrix                # (B, H)
        return torch.cat(
            [base[:, :1], mixed.unsqueeze(1), base[:, 2:]], dim=1
        )

    def _forward(input_ids, attention_mask, emo_probs):
        """One forward pass, routing through inputs_embeds only when the emotion
        channel is in use. text_only_finetuned keeps the plain input_ids path so
        the two arms differ in the emotion signal and nothing else."""
        if not use_emotion:
            return model(input_ids=input_ids, attention_mask=attention_mask)
        return model(
            inputs_embeds=_soft_inputs_embeds(input_ids, emo_probs),
            attention_mask=attention_mask,
        )

    # Tokenize datasets with batching
    def tokenize_texts(texts):
        return tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

    train_encoded = tokenize_texts(train_texts_aug)
    val_encoded = tokenize_texts(val_texts_aug)
    test_encoded = tokenize_texts(test_texts_aug)

    # Move all tensors in batch dicts to device
    for key in train_encoded:
        train_encoded[key] = train_encoded[key].to(device)
    for key in val_encoded:
        val_encoded[key] = val_encoded[key].to(device)
    for key in test_encoded:
        test_encoded[key] = test_encoded[key].to(device)

    # Convert labels to long tensors on device
    train_labels_tensor = torch.tensor(train_labels, dtype=torch.long, device=device)
    val_labels_tensor = torch.tensor(val_labels, dtype=torch.long, device=device)

    # Training setup
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    # Class-weighted loss, all-or-nothing via the module-level CLASS_WEIGHTING
    # switch in common. Weights are computed on TRAIN labels
    # only and never touch val/test.
    if _common.CLASS_WEIGHTING:
        w = torch.tensor(class_weights(train_labels), dtype=torch.float32, device=device)
        criterion = torch.nn.CrossEntropyLoss(weight=w)
    else:
        criterion = torch.nn.CrossEntropyLoss()

    batch_size = 32
    num_epochs = _common.MAX_FINETUNE_EPOCHS

    best_val_f1 = -1.0
    best_state_dict = None
    best_val_logits = None

    n_train = len(train_texts_aug)
    generator = torch.Generator().manual_seed(seed)

    train_emo_t = torch.tensor(train_emo_probs, dtype=torch.float32, device=device)
    val_emo_t = torch.tensor(val_emo_probs, dtype=torch.float32, device=device)
    test_emo_t = torch.tensor(test_emo_probs, dtype=torch.float32, device=device)

    def forward_logits(encoded, emo_t, eval_batch_size: int = 128) -> np.ndarray:
        """Batched inference returning raw logits (N, 3)."""
        model.eval()
        chunks = []
        with torch.no_grad():
            for start in range(0, encoded["input_ids"].shape[0], eval_batch_size):
                stop = start + eval_batch_size
                out = _forward(
                    encoded["input_ids"][start:stop],
                    encoded["attention_mask"][start:stop],
                    emo_t[start:stop],
                )
                chunks.append(out.logits.cpu().numpy())
        return np.concatenate(chunks, axis=0)

    def predict(encoded, emo_t, eval_batch_size: int = 128) -> np.ndarray:
        """Batched inference. Never run a whole split in one forward pass —
        the test split is ~1.5k sequences and would spike GPU memory."""
        return np.argmax(forward_logits(encoded, emo_t, eval_batch_size), axis=1)

    # Training loop
    for epoch in range(num_epochs):
        model.train()

        # Shuffle every epoch. The dataset is ordered by seed group, so without
        # this each batch would be near-duplicate paraphrases of one scenario —
        # correlated batches, unstable gradients, and an effectively much
        # smaller batch diversity than 32.
        perm = torch.randperm(n_train, generator=generator).to(device)

        for i in range(0, n_train, batch_size):
            idx = perm[i : i + batch_size]

            optimizer.zero_grad()
            outputs = _forward(
                train_encoded["input_ids"][idx],
                train_encoded["attention_mask"][idx],
                train_emo_t[idx],
            )
            loss = criterion(outputs.logits, train_labels_tensor[idx])
            loss.backward()
            optimizer.step()

        # Validation phase — model selection happens here, never on test
        val_logits = forward_logits(val_encoded, val_emo_t)
        val_preds = np.argmax(val_logits, axis=1)
        val_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state_dict = copy.deepcopy(model.state_dict())
            best_val_logits = val_logits

        print(f"Epoch {epoch+1}/{num_epochs} - Val F1: {val_f1:.4f}")

    # Restore best state dict before test inference
    # Hand the trained model to the driver if it asked for one (no-op
    # during sweeps, which train hundreds of throwaway models).
    _common.offer_checkpoint(
        f"early_or_textft_{encoder_name}_{seed}", best_state_dict, best_val_f1,
        meta={"method": "early_or_textft", "encoder": encoder_name, "seed": seed,
              "regime": _common.EMOTION_REGIME,
              "class_weighting": _common.CLASS_WEIGHTING},
    )
    model.load_state_dict(best_state_dict)

    test_logits = forward_logits(test_encoded, test_emo_t)
    y_pred_test = np.argmax(test_logits, axis=1)

    return (
        test_labels.astype(np.int64),
        y_pred_test.astype(np.int64),
        best_val_logits,
        test_logits,
    )


def run_early_logits(splits, emb, seed, device, encoder_name):
    """
    Same computation as `run_early` but also returns the val/test logits, for
    the post-hoc class-bias correction step (no retraining involved — this
    just exposes what `run_early` already computed).

    Returns (y_val, val_logits, y_test, test_logits).
    """
    set_seed(seed)
    from common import y_of as _y_of

    y_test, _y_pred, val_logits, test_logits = _run_early_or_textft(
        splits, emb, seed, device, encoder_name, use_emotion=True
    )
    y_val = _y_of(splits["val"])
    return y_val, val_logits, y_test, test_logits

In [ ]:
%%writefile text_ft.py
"""
text_ft.py — text-only, FINE-TUNED control.

Deliverable #1 of the extension: isolates "the encoder was fine-tuned" from
"emotion was visible at the input" (early fusion's actual claim).

`run_text_only` (report.py) freezes the encoder and only trains
a LogisticRegression head on top — so it differs from `run_early` in BOTH
(a) no emotion at input AND (b) no fine-tuning. That confounds the two
questions "does fusion help" and "does fine-tuning help". This module fixes
one variable: same fine-tuning as `run_early`, but the input string is just
`text` — no `[EMOTION]` prefix anywhere.

Implemented as a thin wrapper around `early._run_early_or_textft`
with `use_emotion=False`, so the training loop, hyperparameters, and early
stopping are byte-for-byte the same code path as `run_early` — the only
degree of freedom is the input string. This module intentionally imports
from `early` (not just `common`) to guarantee that
identity; duplicating the loop would risk the two silently drifting apart
after a future edit to one but not the other.
"""

import numpy as np
from common import y_of
from early import _run_early_or_textft


def run_text_only_finetuned(splits, emb, seed, device, encoder_name):
    """
    Text-only baseline with a FINE-TUNED encoder (as opposed to
    `run_text_only`, which uses a frozen embedding + LogisticRegression).

    Identical to `run_early` in every respect (hyperparameters, optimizer,
    batch size, epochs, early stopping, class-weighting switch) except the
    input string has no emotion token prepended — plain `text` only.

    Args:
        splits: dict with "train", "val", "test" lists of Row dicts
        emb: dict with embeddings (ignored — this method builds its own
             representation via fine-tuning, like run_early)
        seed: random seed
        device: torch device string (e.g., "cuda" or "cpu")
        encoder_name: "minilm" or "bert", key to ENCODER_IDS

    Returns:
        tuple[np.ndarray, np.ndarray]: (y_true_test, y_pred_test),
            both int arrays of shape (N_test,) with values in 0..2 indexing LABELS
    """
    y_true, y_pred, _val_logits, _test_logits = _run_early_or_textft(
        splits, emb, seed, device, encoder_name, use_emotion=False
    )
    return y_true, y_pred


def run_text_only_finetuned_logits(splits, emb, seed, device, encoder_name):
    """
    Same computation as `run_text_only_finetuned` but also returns val/test
    logits for the post-hoc class-bias correction step.

    Returns (y_val, val_logits, y_test, test_logits).
    """
    y_test, _y_pred, val_logits, test_logits = _run_early_or_textft(
        splits, emb, seed, device, encoder_name, use_emotion=False
    )
    y_val = y_of(splits["val"])
    return y_val, val_logits, y_test, test_logits

In [ ]:
%%writefile sweep.py
"""
sweep.py — equal-budget hyperparameter sweep.

Deliverable #3 of the extension. Applies to `late` and `intermediate` ONLY,
per the spec — `early` and `text_only_finetuned` get NO sweep and run at
standard fine-tuning defaults, which is recorded explicitly in
`ASYMMETRIC_SWEEP_NOTE` below so the asymmetry is stated on a slide rather
than silently assumed. (`film.py` and `attn.py`
also use this module for their sweep, at the SAME grid/budget as
`intermediate`, per their own module docstrings — they are new intermediate-
style methods, not part of the mandatory late/intermediate comparison, but
"equal footing" for them means the same grid size, not a bigger one.)

Grid (identical size for every method swept — 3 x 3 x 2 = 18 configs):
    lr      in {3e-4, 1e-3, 3e-3}
    hidden  in {128, 256, 512}   (late: text-branch hidden size; intermediate
                                  / film / attn: the fc1 width after fusion)
    dropout in {0.1, 0.3}

Selection: winning config = highest VAL macro-F1 (using one search seed).
Then the winner is re-run at 3 seeds and reported on TEST — never select on
test.

These methods train on frozen embeddings in seconds (no encoder fine-tuning
involved), so the sweep runs on the FULL training split — no subsampling.
`_run_late_core` / `_run_intermediate_core` / `_run_film_core` / `_run_attn_core`
already train on `splits["train"]` directly; this module does not subsample it.
"""

from __future__ import annotations

import itertools
import numpy as np
from sklearn.metrics import f1_score

from common import make_result

ASYMMETRIC_SWEEP_NOTE = (
    "Sweep coverage is intentionally asymmetric: late/intermediate/film/attn "
    "(frozen-embedding methods, seconds per run) get an 18-config grid search "
    "selected on val macro-F1. early and text_only_finetuned (fine-tune the "
    "encoder, minutes per run) get NO sweep and run at the standard "
    "fine-tuning defaults (AdamW lr=2e-5, batch 32, 4 epochs) specified in "
    "the original contract. State this on the slide: an early/text_only_ft "
    "win over a swept late/intermediate is a comparison of a tuned baseline "
    "against an untuned one, in early's favour if anything — so it is a "
    "conservative, not inflated, comparison."
)

LR_GRID = [3e-4, 1e-3, 3e-3]
HIDDEN_GRID = [128, 256, 512]
DROPOUT_GRID = [0.1, 0.3]


def sweep_grid() -> list[dict]:
    """The 18 (lr, hidden, dropout) configs, identical for every swept method."""
    return [
        {"lr": lr, "hidden": hidden, "dropout": dropout}
        for lr, hidden, dropout in itertools.product(LR_GRID, HIDDEN_GRID, DROPOUT_GRID)
    ]


def _core_val_macro_f1(core_result: dict) -> float:
    return float(f1_score(core_result["y_val"], core_result["val_pred"],
                          average="macro", zero_division=0))


def sweep_method(
    core_fn,
    hparam_names: dict,
    splits: dict,
    emb: dict,
    device: str,
    encoder_name: str,
    search_seed: int = 0,
    eval_seeds: tuple[int, ...] = (0, 1, 2),
) -> dict:
    """
    Generic equal-budget sweep runner.

    `core_fn(splits, emb, seed, device, encoder_name, **kwargs) -> dict` must
    be one of `_run_late_core` / `_run_intermediate_core` / `_run_film_core` /
    `_run_attn_core` (or anything with that shape: returns a dict with
    y_val/val_pred/y_test/test_pred/test_logits/val_logits keys).

    `hparam_names` maps the grid's generic keys {"lr","hidden","dropout"} to
    the specific keyword `core_fn` expects — e.g. for late's core,
    {"lr": "lr", "hidden": "text_hidden", "dropout": "text_dropout"}, since
    late's hidden size is a "text_hidden" kwarg, not "hidden".

    Runs the full 18-config grid ONCE at `search_seed`, selects by val
    macro-F1, then re-runs the winning config at every seed in `eval_seeds`
    and returns those as the reportable test results.

    Returns:
        {
          "best_config": {"lr":.., "hidden":.., "dropout":..},
          "best_val_macro_f1": float,
          "all_configs": [{"config":.., "val_macro_f1":..}, ...],   # all 18, for the appendix
          "test_runs": [ (y_true, y_pred, seed), ... ],             # one per eval_seed, on the winner
        }
    """
    all_configs = []
    best_val_f1 = -1.0
    best_config = None

    for cfg in sweep_grid():
        kwargs = {hparam_names[k]: v for k, v in cfg.items()}
        result = core_fn(splits, emb, search_seed, device, encoder_name, **kwargs)
        val_f1 = _core_val_macro_f1(result)
        all_configs.append({"config": dict(cfg), "val_macro_f1": val_f1})
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_config = dict(cfg)

    test_runs = []
    kwargs = {hparam_names[k]: v for k, v in best_config.items()}
    for seed in eval_seeds:
        result = core_fn(splits, emb, seed, device, encoder_name, **kwargs)
        test_runs.append((result["y_test"], result["test_pred"], seed))

    return {
        "best_config": best_config,
        "best_val_macro_f1": best_val_f1,
        "all_configs": all_configs,
        "test_runs": test_runs,
    }


def sweep_late(splits, emb, device, encoder_name, search_seed=0, eval_seeds=(0, 1, 2)) -> dict:
    """Sweep `late`'s text branch: hidden -> text_hidden, dropout -> text_dropout."""
    from late import _run_late_core
    return sweep_method(
        _run_late_core,
        {"lr": "lr", "hidden": "text_hidden", "dropout": "text_dropout"},
        splits, emb, device, encoder_name, search_seed, eval_seeds,
    )


def sweep_intermediate(splits, emb, device, encoder_name, search_seed=0, eval_seeds=(0, 1, 2)) -> dict:
    """Sweep `intermediate`'s fc1 width/dropout/lr."""
    from inter import _run_intermediate_core
    return sweep_method(
        _run_intermediate_core,
        {"lr": "lr", "hidden": "hidden", "dropout": "dropout"},
        splits, emb, device, encoder_name, search_seed, eval_seeds,
    )


def sweep_film(splits, emb, device, encoder_name, search_seed=0, eval_seeds=(0, 1, 2)) -> dict:
    """Sweep `intermediate_film`'s fc1 width/dropout/lr, same grid as intermediate."""
    from film import _run_film_core
    return sweep_method(
        _run_film_core,
        {"lr": "lr", "hidden": "hidden", "dropout": "dropout"},
        splits, emb, device, encoder_name, search_seed, eval_seeds,
    )


def sweep_attn(splits, emb, device, encoder_name, search_seed=0, eval_seeds=(0, 1, 2)) -> dict:
    """Sweep `intermediate_attn`'s fc1 width/dropout/lr, same grid as intermediate."""
    from attn import _run_attn_core
    return sweep_method(
        _run_attn_core,
        {"lr": "lr", "hidden": "hidden", "dropout": "dropout"},
        splits, emb, device, encoder_name, search_seed, eval_seeds,
    )


def sweep_results_to_records(method: str, encoder: str, variant: str, sweep_out: dict) -> list[dict]:
    """Convert a `sweep_*` output's `test_runs` into the flat result-dict
    format (`common.make_result`) used everywhere else, so the
    sweep winner slots directly into `results_table` / `plot_results` next
    to the unswept methods."""
    return [
        make_result(method, encoder, variant, seed, y_true, y_pred)
        for y_true, y_pred, seed in sweep_out["test_runs"]
    ]


def json_safe(obj):
    """Recursively convert numpy types so a sweep result can be json.dump'd.

    `sweep_method` returns `test_runs` as raw numpy arrays because the driver
    wants them in memory; that makes the dict itself un-serialisable. Rather
    than degrading the in-memory return value, convert at the point of writing.
    """
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    return obj


def sweep_summary(sweep_out: dict) -> dict:
    """A compact, JSON-safe view of one sweep: the winner, the full config
    ranking (useful as a slide appendix), and per-seed test scores on the
    winner — with the raw prediction arrays dropped."""
    from common import compute_metrics

    return {
        "best_config": json_safe(sweep_out["best_config"]),
        "best_val_macro_f1": float(sweep_out["best_val_macro_f1"]),
        "all_configs": json_safe(sweep_out["all_configs"]),
        "n_configs": len(sweep_out["all_configs"]),
        "winner_test_per_seed": [
            {"seed": int(seed), **{k: v for k, v in compute_metrics(y_true, y_pred).items()
                                   if k != "confusion"}}
            for y_true, y_pred, seed in sweep_out["test_runs"]
        ],
    }

In [ ]:
%%writefile report.py
import matplotlib
matplotlib.use("Agg")  # Headless backend; must come before pyplot import
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

import common as _common
from common import LABELS, y_of, emotion_features, set_seed
# CLASS_WEIGHTING read live off `_common.CLASS_WEIGHTING` — see
# early.py's comment for why (import-time binding would freeze
# the flag and ignore later toggles from the driver notebook).


def run_majority(splits, emb, seed, device, encoder_name):
    """Majority class baseline. Predicts train-set majority class for all test rows.

    Ignores emb, device, encoder_name; accepted for signature uniformity.

    Args:
        splits: dict with 'train', 'val', 'test' keys containing Row lists
        emb: dict (ignored)
        seed: random seed
        device: device string (ignored)
        encoder_name: encoder name (ignored)

    Returns:
        (y_true_test, y_pred_test) both shape (N_test,) with values in {0,1,2}
    """
    set_seed(seed)

    # Get majority class from train split only
    y_train = y_of(splits["train"])
    majority_class = np.bincount(y_train).argmax()

    # Get true labels for test
    y_test = y_of(splits["test"])

    # Predict majority class for all test rows
    y_pred_test = np.full(len(y_test), majority_class, dtype=np.int64)

    return y_test, y_pred_test


def run_emotion_only(splits, emb, seed, device, encoder_name):
    """Emotion-only baseline. Trains LogisticRegression on 6-dim one-hot emotion features.

    Ignores emb, device, encoder_name; accepted for signature uniformity.
    Fits on TRAIN split, predicts on TEST split.

    Args:
        splits: dict with 'train', 'val', 'test' keys containing Row lists
        emb: dict (ignored)
        seed: random seed for LogisticRegression
        device: device string (ignored)
        encoder_name: encoder name (ignored)

    Returns:
        (y_true_test, y_pred_test) both shape (N_test,) with values in {0,1,2}
    """
    set_seed(seed)

    # Get one-hot emotion features and labels
    X_train = emotion_features(splits["train"], "train", seed)
    y_train = y_of(splits["train"])
    X_test = emotion_features(splits["test"], "test", seed)
    y_test = y_of(splits["test"])

    # Fit LogisticRegression on train, predict on test. class_weight="balanced"
    # honours the same all-or-nothing CLASS_WEIGHTING switch used by the
    # torch-trained methods (common.CLASS_WEIGHTING).
    model = LogisticRegression(
        max_iter=1000, random_state=seed,
        class_weight=("balanced" if _common.CLASS_WEIGHTING else None),
    )
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)

    return y_test, y_pred_test


def run_emotion_only_logits(splits, emb, seed, device, encoder_name):
    """Same computation as `run_emotion_only`, also returning val/test
    decision-function scores (pseudo-logits) for the post-hoc class-bias
    correction step. No retraining."""
    set_seed(seed)
    X_train = emotion_features(splits["train"], "train", seed)
    y_train = y_of(splits["train"])
    X_val = emotion_features(splits["val"], "val", seed)
    y_val = y_of(splits["val"])
    X_test = emotion_features(splits["test"], "test", seed)
    y_test = y_of(splits["test"])

    model = LogisticRegression(
        max_iter=1000, random_state=seed,
        class_weight=("balanced" if _common.CLASS_WEIGHTING else None),
    )
    model.fit(X_train, y_train)
    val_logits = model.decision_function(X_val)
    test_logits = model.decision_function(X_test)
    return y_val, val_logits, y_test, test_logits


def run_text_only(splits, emb, seed, device, encoder_name):
    """Text-only baseline. Trains LogisticRegression on frozen text embeddings.

    Uses emb['train'] and emb['test']; ignores device, encoder_name.
    Fits on TRAIN split, predicts on TEST split.

    Args:
        splits: dict with 'train', 'val', 'test' keys containing Row lists
        emb: dict with 'train', 'val', 'test' embeddings (N, D) float32
        seed: random seed for LogisticRegression
        device: device string (ignored)
        encoder_name: encoder name (ignored)

    Returns:
        (y_true_test, y_pred_test) both shape (N_test,) with values in {0,1,2}
    """
    set_seed(seed)

    # Get text embeddings and labels
    X_train = emb["train"]
    y_train = y_of(splits["train"])
    X_test = emb["test"]
    y_test = y_of(splits["test"])

    # Fit LogisticRegression on train, predict on test. class_weight="balanced"
    # honours the same all-or-nothing CLASS_WEIGHTING switch used by the
    # torch-trained methods (common.CLASS_WEIGHTING).
    model = LogisticRegression(
        max_iter=1000, random_state=seed,
        class_weight=("balanced" if _common.CLASS_WEIGHTING else None),
    )
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)

    return y_test, y_pred_test


def run_text_only_logits(splits, emb, seed, device, encoder_name):
    """Same computation as `run_text_only`, also returning val/test
    decision-function scores (pseudo-logits) for the post-hoc class-bias
    correction step. No retraining."""
    set_seed(seed)
    X_train = emb["train"]
    y_train = y_of(splits["train"])
    X_val = emb["val"]
    y_val = y_of(splits["val"])
    X_test = emb["test"]
    y_test = y_of(splits["test"])

    model = LogisticRegression(
        max_iter=1000, random_state=seed,
        class_weight=("balanced" if _common.CLASS_WEIGHTING else None),
    )
    model.fit(X_train, y_train)
    val_logits = model.decision_function(X_val)
    test_logits = model.decision_function(X_test)
    return y_val, val_logits, y_test, test_logits


def tune_class_bias(val_logits: np.ndarray, y_val: np.ndarray,
                    n_passes: int = 3, grid_points: int = 41,
                    lo: float = -3.0, hi: float = 3.0) -> np.ndarray:
    """
    Post-hoc class-bias correction (deliverable #4): fit a per-class additive
    logit offset (3 scalars, one per LABELS index) that maximises VAL macro-F1
    when added to `val_logits` before argmax. No retraining — this only
    shifts the decision boundary of an already-trained model.

    Method: coordinate search. For each of `n_passes` passes over the 3
    classes, grid-search that class's offset over `grid_points` values in
    [lo, hi] holding the other two fixed at their current best, keep whichever
    value improves val macro-F1 (ties keep the earlier, i.e. smaller-offset,
    value). This is intentionally simple/greedy rather than an exact 3-D
    optimum, in keeping with a POST-HOC correction — the point is to move the
    class-imbalance collapse (f1_borderline==0.0) off dead zero, not to
    squeeze out the last 0.1% of macro-F1.

    `val_logits` may be true logits (torch model output) or decision-function
    scores (sklearn LogisticRegression) — both are pre-argmax, real-valued
    per-class scores and the offset acts identically on either.

    Args:
        val_logits: (N_val, 3) real-valued scores, one column per LABELS index
        y_val: (N_val,) int true labels in 0..2
        n_passes: number of full coordinate-descent sweeps over the 3 classes
        grid_points: number of candidate offsets tried per class per pass
        lo, hi: search range for the additive offset

    Returns:
        (3,) float32 array of per-class additive offsets, indexed like LABELS.
    """
    n_classes = val_logits.shape[1]
    biases = np.zeros(n_classes, dtype=np.float64)
    grid = np.linspace(lo, hi, grid_points)

    def macro_f1_with(b):
        preds = np.argmax(val_logits + b, axis=1)
        return f1_score(y_val, preds, average="macro", zero_division=0)

    best_f1 = macro_f1_with(biases)
    for _ in range(n_passes):
        improved = False
        for c in range(n_classes):
            best_val_for_c = biases[c]
            local_best_f1 = best_f1
            for g in grid:
                trial = biases.copy()
                trial[c] = g
                f1 = macro_f1_with(trial)
                if f1 > local_best_f1:
                    local_best_f1 = f1
                    best_val_for_c = g
            if local_best_f1 > best_f1:
                biases[c] = best_val_for_c
                best_f1 = local_best_f1
                improved = True
        if not improved:
            break

    return biases.astype(np.float32)


def apply_class_bias(logits: np.ndarray, biases: np.ndarray) -> np.ndarray:
    """Apply per-class additive offsets from `tune_class_bias` to a logits
    array before argmax. Pure function, no state, no retraining."""
    return logits + biases


def bias_corrected_predictions(val_logits, y_val, test_logits):
    """
    Convenience wrapper: fit the bias offsets on val, apply to test, return
    (uncorrected_test_pred, corrected_test_pred, biases) so callers can
    report both numbers side by side, as the spec requires.
    """
    biases = tune_class_bias(val_logits, y_val)
    uncorrected_pred = np.argmax(test_logits, axis=1)
    corrected_pred = np.argmax(apply_class_bias(test_logits, biases), axis=1)
    return uncorrected_pred, corrected_pred, biases


def results_table(results):
    """Aggregate result dicts by (method, encoder, variant) and compute summary statistics.

    Args:
        results: list of result dicts from run_* methods, each containing
                 'method', 'encoder', 'variant', 'seed', and metric keys
                 (acc, macro_f1, f1_normal, f1_borderline, f1_anomaly, confusion)

    Returns:
        pd.DataFrame with one row per (method, encoder, variant) and columns:
        method, encoder, variant, acc_mean, acc_std, macro_f1_mean, macro_f1_std,
        f1_normal_mean, f1_borderline_mean, f1_anomaly_mean, n_seeds.
        Sorted by variant, encoder, macro_f1_mean descending.
    """
    # Convert to DataFrame for easier grouping
    df = pd.DataFrame(results)

    # Group by (method, encoder, variant) and aggregate
    grouped = df.groupby(["method", "encoder", "variant"], as_index=False).agg({
        "acc": ["mean", "std"],
        "macro_f1": ["mean", "std"],
        "f1_normal": "mean",
        "f1_borderline": "mean",
        "f1_anomaly": "mean",
        "seed": "count"
    })

    # Flatten multi-level column names
    grouped.columns = ["method", "encoder", "variant",
                       "acc_mean", "acc_std",
                       "macro_f1_mean", "macro_f1_std",
                       "f1_normal_mean", "f1_borderline_mean", "f1_anomaly_mean",
                       "n_seeds"]

    # Sort by variant, encoder, macro_f1_mean descending
    grouped = grouped.sort_values(
        by=["variant", "encoder", "macro_f1_mean"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    return grouped


def plot_results(df, out_path):
    """Plot grouped bar chart of macro-F1 scores with error bars.

    Creates one subplot per (encoder, variant) combination, with methods on x-axis,
    grouped bars showing macro_f1_mean with macro_f1_std error bars, and a horizontal
    dashed reference line for the majority baseline in each panel.

    Args:
        df: DataFrame from results_table() with aggregated metrics
        out_path: output path for the PNG file (e.g., 'results_plot.png')
    """
    # Get unique (encoder, variant) combinations, sorted
    encoder_variants = df[["encoder", "variant"]].drop_duplicates().sort_values(
        by=["variant", "encoder"]
    ).reset_index(drop=True)

    n_plots = len(encoder_variants)
    n_cols = 2
    n_rows = (n_plots + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1 or n_cols == 1:
        axes = axes.reshape(n_rows, n_cols)

    # Get majority baseline macro_f1 (should be same for all encoder/variant)
    majority_rows = df[df["method"] == "majority"]
    if len(majority_rows) > 0 and np.isfinite(majority_rows.iloc[0]["macro_f1_mean"]):
        majority_f1 = float(majority_rows.iloc[0]["macro_f1_mean"])
    else:
        majority_f1 = 0.3  # fallback

    # Get shared y-limits across all subplots
    y_min = 0
    # pandas .std() over a single observation is NaN, so any 1-seed run (a
    # pre-flight, or a deliberately cheap sweep) would otherwise propagate NaN
    # into set_ylim and raise "Axis limits cannot be NaN or Inf". Treat a missing
    # spread as zero spread — with one seed there genuinely is no measured spread.
    means = df["macro_f1_mean"].fillna(0.0)
    stds = df["macro_f1_std"].fillna(0.0)
    y_max = float((means + stds).max()) + 0.05
    if not np.isfinite(y_max) or y_max <= y_min:
        y_max = 1.0

    for plot_idx, (_, enc_var_row) in enumerate(encoder_variants.iterrows()):
        row_idx = plot_idx // n_cols
        col_idx = plot_idx % n_cols
        ax = axes[row_idx, col_idx]

        encoder = enc_var_row["encoder"]
        variant = enc_var_row["variant"]

        # Filter results for this (encoder, variant)
        subset = df[(df["encoder"] == encoder) & (df["variant"] == variant)].copy()
        subset = subset.sort_values("method")  # consistent ordering

        # Prepare bar data
        methods = subset["method"].values
        # Same NaN guard as the y-limit above: a 1-seed run has no measured
        # spread, and matplotlib's yerr also refuses NaN.
        means = subset["macro_f1_mean"].fillna(0.0).values
        stds = subset["macro_f1_std"].fillna(0.0).values

        # Plot grouped bars
        x_pos = np.arange(len(methods))
        ax.bar(x_pos, means, yerr=stds, capsize=5, alpha=0.7, color="steelblue", edgecolor="black")

        # Add reference line for majority baseline
        ax.axhline(y=majority_f1, color="red", linestyle="--", linewidth=2, label="Majority baseline")

        # Formatting
        ax.set_xlabel("Method", fontsize=10)
        ax.set_ylabel("Macro F1", fontsize=10)
        ax.set_title(f"Encoder: {encoder}, Variant: {variant}", fontsize=11, fontweight="bold")
        ax.set_xticks(x_pos)
        ax.set_xticklabels(methods, rotation=45, ha="right")
        ax.set_ylim(y_min, y_max)
        ax.grid(axis="y", alpha=0.3, linestyle=":")
        ax.legend(fontsize=9)

    # Hide unused subplots
    for plot_idx in range(n_plots, n_rows * n_cols):
        row_idx = plot_idx // n_cols
        col_idx = plot_idx % n_cols
        axes[row_idx, col_idx].set_visible(False)

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_confusion(results, method, encoder, variant, out_path):
    """Plot confusion matrix heatmap for a specific (method, encoder, variant).

    Sums confusion matrices across all run seeds for the requested combination,
    renders as a heatmap with annotated cell counts.

    Args:
        results: list of result dicts (flat, as returned by run_* methods)
        method: method name (e.g., 'majority', 'late', 'intermediate')
        encoder: encoder name (e.g., 'minilm', 'bert')
        variant: variant name (e.g., 'full', 'filtered')
        out_path: output path for the PNG file

    Raises:
        ValueError: if no results match the (method, encoder, variant) filter
    """
    # Filter results
    matching = [r for r in results
                if r["method"] == method and r["encoder"] == encoder and r["variant"] == variant]

    if not matching:
        raise ValueError(f"No results found for method={method}, encoder={encoder}, variant={variant}")

    # Sum confusion matrices across seeds
    confusion_sum = None
    for r in matching:
        conf = np.array(r["confusion"], dtype=np.int64)
        if confusion_sum is None:
            confusion_sum = conf.copy()
        else:
            confusion_sum += conf

    # Plot heatmap
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(confusion_sum, cmap="Blues", aspect="auto")

    # Add text annotations
    for i in range(confusion_sum.shape[0]):
        for j in range(confusion_sum.shape[1]):
            # Choose text color based on background intensity
            threshold = confusion_sum.max() / 2
            text_color = "white" if confusion_sum[i, j] > threshold else "black"
            ax.text(j, i, str(int(confusion_sum[i, j])), ha="center", va="center",
                   color=text_color, fontsize=14, fontweight="bold")

    # Set ticks and labels
    ax.set_xticks(np.arange(len(LABELS)))
    ax.set_yticks(np.arange(len(LABELS)))
    ax.set_xticklabels(LABELS, fontsize=11)
    ax.set_yticklabels(LABELS, fontsize=11)
    ax.set_xlabel("Predicted", fontsize=12, fontweight="bold")
    ax.set_ylabel("True", fontsize=12, fontweight="bold")
    ax.set_title(f"Confusion Matrix: {method} (encoder={encoder}, variant={variant})",
                fontsize=13, fontweight="bold", pad=20)

    plt.colorbar(im, ax=ax, label="Count")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()

In [ ]:
import importlib, sys
for m in ["common","ser_noise","encoders",
          "late","inter","film","attn",
          "early","text_ft","sweep","report"]:
    sys.modules.pop(m, None)

import common as C
import ser_noise as SER
from encoders import get_embeddings, get_token_embeddings
from late import run_late
from inter import run_intermediate
from film import run_intermediate_film
from attn import run_intermediate_attn
from early import run_early
from text_ft import run_text_only_finetuned
from sweep import sweep_late, sweep_intermediate
from report import (run_majority, run_emotion_only, run_text_only,
                                 results_table, plot_results, plot_confusion,
                                 tune_class_bias, apply_class_bias)
# intermediate_attn builds its own token cache internally, so point it at Drive
# or the ~2 GB cache is rebuilt from scratch every session.
C.TOKEN_CACHE_DIR = TOK_CACHE_DIR
print("modules loaded | token cache ->", C.TOKEN_CACHE_DIR)

### The HuggingFace loading warning is expected — read this once

Loading `bert-base-uncased` for sequence classification prints:

```
cls.predictions.*, cls.seq_relationship.*   UNEXPECTED
classifier.weight, classifier.bias          MISSING
```

Both are normal. `cls.*` are the pretraining MLM/NSP heads, which have no home in a
sequence-classification model and are discarded. `classifier.*` is the 3-way head we are about to
train, so it *must* start uninitialised.

But the same message would also appear if the **encoder body itself** failed to load, which would
silently train a randomly-initialised BERT. So `early.py` does not rely on eyeballing
the log — it loads with `output_loading_info=True` and asserts that nothing outside
`classifier` / `pooler` is missing.

## 3. Smoke test — split integrity

In [ ]:
C.self_test(DATASET_PATH)

## 4. The SER noise model

The emotion channel used everywhere above is `gen_emotion`, an **oracle label**. Deployment does not
have that: it has the SER's output, and the SER is wrong a substantial fraction of the time. In this
task that is not ordinary noise — the anomaly label IS an arousal/content mismatch, so a flipped
arousal reading **inverts the verdict** rather than degrading it.

The numbers below are measured, not invented: they come from the SER voice-risk confusion matrix on
the held-out speaker-independent validation split (n=902) and reconcile exactly with the three
reported SER-channel figures (accuracy 86.0%, precision 95.9%, recall 83.1%).

In [ ]:
print(SER.summary())
print()
rows_all = C.load_rows(DATASET_PATH, "full")
emos = [r["gen_emotion"] for r in rows_all]
for s in (0, 1, 2):
    print(f"  seed {s}: {SER.flip_report(emos, SER.simulate(emos, s))}")
print()
print("~13% of utterances receive an INVERTED arousal reading from the real SER.")
print("Asymmetry that matters for the demo: 'alarmed voice + trivial content' anomalies")
print("survive only 83.1% of the time, 'calm voice + severe content' survive 92.3%.")
print("=> for a live demo, prefer the calm-voice-severe-content direction.")

## 5. Run plan

The full cross product would be 3 regimes x 2 weightings x 11 methods x 2 encoders x 2 variants x
3 seeds. The fine-tuning methods (`early`, `text_only_finetuned`) dominate the cost at ~5 min per
run, so the cross product is several GPU-days. The plan below is pruned deliberately:

| phase | regime | weighting | variants | methods | question it answers |
|---|---|---|---|---|---|
| 1 | oracle | off | full + filtered | all | the v1 baseline, reproduced |
| 2 | oracle | **on** | full + filtered | all | does fixing the borderline collapse change the ranking? |
| 3 | ser_test | on | full | emotion-dependent only | how far does the current approach fall at deployment? |
| 4 | ser_both | on | full | emotion-dependent only | does training under SER noise close that gap? |

Phases 3 and 4 skip `majority`, `text_only` and `text_only_finetuned` because none of them read the
emotion channel, so the regime cannot affect them — running them would burn GPU on a guaranteed
no-op. They also skip the `filtered` variant: that ablation's job (does generator drift change the
ranking?) was already answered in v1 with the ranking preserved.

Phases 3 and 4 share the SAME test condition, so comparing them is fair. Phase 2 is the ceiling
above both.

In [ ]:
import sweep as SW

ALL_METHODS = {
    "majority":            run_majority,
    "emotion_only":        run_emotion_only,
    "text_only":           run_text_only,
    "text_only_finetuned": run_text_only_finetuned,
    "late":                run_late,
    "intermediate":        run_intermediate,
    "intermediate_film":   run_intermediate_film,
    "intermediate_attn":   run_intermediate_attn,
    "early":               run_early,
}

# Methods that actually consume the emotion channel. The others are invariant to
# EMOTION_REGIME by construction, so phases 3-4 skip them.
EMOTION_DEPENDENT = ["emotion_only", "late", "intermediate",
                     "intermediate_film", "intermediate_attn", "early"]

ENCODERS  = ["minilm", "bert"]
RUN_SEEDS = [0, 1, 2]

RUN_PLAN = [
    dict(name="p1_oracle_unweighted", regime="oracle",   weighted=False,
         variants=["full", "filtered"], methods=list(ALL_METHODS)),
    dict(name="p2_oracle_weighted",   regime="oracle",   weighted=True,
         variants=["full", "filtered"], methods=list(ALL_METHODS)),
    dict(name="p3_sertest_weighted",  regime="ser_test", weighted=True,
         variants=["full"],             methods=EMOTION_DEPENDENT),
    dict(name="p4_serboth_weighted",  regime="ser_both", weighted=True,
         variants=["full"],             methods=EMOTION_DEPENDENT),
]

# --- PRE-FLIGHT --------------------------------------------------------------
# "off"   : the real thing (4 phases, ~5 h on a T4 including caches).
# "fast"  : plumbing only. Cheap methods, minilm, 1 seed, 2-config sweep.
#           ~3-5 min. Proves the dataset path, Drive mount, module writeout,
#           split integrity, driver loop, bias correction, tables, plots and —
#           most importantly — THE DRIVE SAVE all work.
# "paths" : every CODE PATH, cheaply. Adds early / text_only_finetuned /
#           intermediate_attn and the bert encoder, but 1 fine-tune epoch and
#           1 seed. ~15-20 min. This is the one that protects an overnight run:
#           "fast" deliberately skips the three newest and riskiest branches
#           (soft inputs_embeds early fusion, the token-level attn cache, the
#           minilm token path that used to return NaN), so a green "fast" run
#           tells you nothing about whether they crash at 3am.
PRE_FLIGHT = "paths"

# Every knob a pre-flight touches is reset HERE, unconditionally, before the
# branches run. Without this the cell is not idempotent: the pre-flight sets
# module-level state on objects that survive in the kernel, so flipping
# PRE_FLIGHT to "off" and re-running would silently keep
# MAX_FINETUNE_EPOCHS = 1 and a 2-config sweep grid — an overnight run that
# looks fine and produces meaningless fine-tuning numbers.
C.MAX_FINETUNE_EPOCHS = 4          # the value the reported results used
SW.LR_GRID      = [3e-4, 1e-3, 3e-3]
SW.HIDDEN_GRID  = [128, 256, 512]
SW.DROPOUT_GRID = [0.1, 0.3]
sweep_winners = {}                  # stale winners from a 2-config grid must not leak

if PRE_FLIGHT == "fast":
    cheap = [m for m in ALL_METHODS if m not in
             ("early", "text_only_finetuned", "intermediate_attn")]
    ENCODERS  = ["minilm"]
    RUN_SEEDS = [0]
    RUN_PLAN  = [dict(name="preflight_fast", regime="oracle", weighted=False,
                      variants=["full"], methods=cheap)]
elif PRE_FLIGHT == "paths":
    RUN_SEEDS = [0]
    C.MAX_FINETUNE_EPOCHS = 1     # exercise the branch, do not try to win with it
    RUN_PLAN  = [
        dict(name="preflight_oracle", regime="oracle", weighted=False,
             variants=["full"], methods=list(ALL_METHODS)),
        dict(name="preflight_serboth", regime="ser_both", weighted=True,
             variants=["full"], methods=EMOTION_DEPENDENT),
    ]

QUICK = PRE_FLIGHT != "off"   # kept for the manifest / older references

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
total = sum(len(p["methods"]) * len(ENCODERS) * len(p["variants"]) * len(RUN_SEEDS)
            for p in RUN_PLAN)
print("device:", DEVICE)
print(f"{len(RUN_PLAN)} phases, {total} runs queued")

## 6. Hyperparameter sweep

Equal 18-config grid (`lr` x `hidden` x `dropout`) for `late` and `intermediate`. Equal budget is not
a nicety: those two carry the headline comparison, and tuning one but not the other would turn a
fusion-level finding into a tuning artifact.

`early` and `text_only_finetuned` get **no** sweep and run at standard fine-tuning defaults
(AdamW 2e-5). State that on the slide — their numbers are a conservative lower bound.

The sweep runs once per weighting setting on the oracle regime and the winners are reused for the
SER-noise phases. Re-sweeping per phase would multiply the cost for a second-order effect; note it
as a limitation rather than pretending it did not happen.

**Known caveat from v1:** the winners kept landing on the EDGE of the grid (lr=0.003 and
hidden=512, both maxima), which usually means the range is too narrow. `WIDEN_GRID` extends it at
roughly +50% sweep time.

In [ ]:
WIDEN_GRID = False   # adds lr=1e-2 and hidden=1024
if WIDEN_GRID:
    # The sweep reads these module-level grids, so widening is a mutation here
    # rather than a call argument. Applies to EVERY swept method equally, which
    # is the whole point — an unequal grid would make "tuned vs untuned"
    # masquerade as a fusion-level result.
    SW.LR_GRID = [3e-4, 1e-3, 3e-3, 1e-2]
    SW.HIDDEN_GRID = [128, 256, 512, 1024]
if PRE_FLIGHT != "off":
    # The sweep is what actually made the old QUICK run take ~30 minutes: it is
    # independent of RUN_PLAN and trained 18 configs x 60 epochs per method.
    # Two configs is enough to prove the sweep mechanics and the winner plumbing.
    SW.LR_GRID = [1e-3]
    SW.HIDDEN_GRID = [256]
    SW.DROPOUT_GRID = [0.1, 0.3]
print("sweep grid:", len(SW.sweep_grid()), "configs per method")

SEED_UNIVERSE = sorted({r["seed_id"] for r in C.load_rows(DATASET_PATH, "full")})

def build_splits(variant):
    """Seed assignment is pinned to the FULL dataset for both variants, so the
    filtered run is evaluated on a SUBSET of the same held-out scenarios rather
    than on a freshly reshuffled split. Without this, any full-vs-filtered gap
    would be split noise instead of a data-quality effect."""
    rows = C.load_rows(DATASET_PATH, variant=variant)
    splits = C.make_splits(rows, seed_universe=SEED_UNIVERSE)
    C.assert_no_seed_overlap(splits)
    return splits

# sweep_winners was reset in the pre-flight cell above, so a stale winner found
# with the shrunken pre-flight grid can never leak into a real run.

In [ ]:
import json

C.EMOTION_REGIME = "oracle"
for weighted in sorted({p["weighted"] for p in RUN_PLAN}):
    C.CLASS_WEIGHTING = weighted
    splits = build_splits("full")
    for encoder in ENCODERS:
        emb = get_embeddings(splits, encoder, DEVICE, cache_dir=CACHE_DIR)
        for method, fn in (("late", sweep_late), ("intermediate", sweep_intermediate)):
            t0 = time.time()
            best = fn(splits, emb, DEVICE, encoder)
            sweep_winners[(weighted, method, encoder)] = best
            print(f"  sweep w={weighted} {method}/{encoder}: {best['best_config']} "
                  f"val_mF1={best['best_val_macro_f1']:.4f}  ({time.time()-t0:.0f}s)")

# sweep_method keeps `test_runs` as raw numpy arrays (the driver wants them in
# memory), which makes the dict itself un-serialisable. sweep_summary() drops the
# arrays and casts the rest.
with open(f"{LOCAL_OUT}/sweep_winners.json", "w") as fh:
    json.dump({f"{k[0]}|{k[1]}|{k[2]}": SW.sweep_summary(v)
               for k, v in sweep_winners.items()}, fh, indent=2)
print("\nsweep done")

## 7. Driver

Phases run **sequentially** in one pass so a single execution produces the whole comparison.
`EMOTION_REGIME` and `CLASS_WEIGHTING` are set on the `common` module object and read
live by every method — they are deliberately NOT imported by name anywhere, because that would
freeze the value at import time and silently keep using a stale setting.

In [ ]:
import traceback
import numpy as np

# Fail loudly rather than burning a night on silently-degraded settings. This
# catches the "flipped PRE_FLIGHT to off but re-ran only some cells" case.
if PRE_FLIGHT == "off":
    assert C.MAX_FINETUNE_EPOCHS == 4, (
        f"MAX_FINETUNE_EPOCHS is {C.MAX_FINETUNE_EPOCHS}, not 4 — a pre-flight value "
        "leaked. Re-run the RUN PLAN cell (or restart the runtime) before the real run.")
    assert len(SW.sweep_grid()) == (18 if not WIDEN_GRID else 32), (
        f"sweep grid has {len(SW.sweep_grid())} configs — a pre-flight value leaked. "
        "Re-run the RUN PLAN cell before the real run.")
    assert RUN_SEEDS == [0, 1, 2], f"RUN_SEEDS={RUN_SEEDS}, expected [0, 1, 2]"
    for _m, _e, _v in [(m, e, p["name"]) for p in RUN_PLAN for m in p["methods"] for e in ENCODERS]:
        break
    print("pre-run checks passed: epochs=4, "
          f"{len(SW.sweep_grid())} sweep configs, seeds={RUN_SEEDS}")

results   = []     # flat result records
logits_by = {}     # (phase, method, encoder, variant, seed) -> (y_val, val_logits, y_test, test_logits)
checkpoints = {}   # tag -> {"state_dict","val_f1","meta"}
failures  = []
t_start = time.time()

# --- tuned wrappers ---------------------------------------------------------
# The sweep was previously computed, saved, and then IGNORED: the driver called
# the plain `run_*` entry points, which use the module defaults. The uniform
# signature (splits, emb, seed, device, encoder_name) has no room for
# hyperparameters, so the winning config has to be closed over instead.
SWEPT_CORES = {}
try:
    from late import _run_late_core
    from inter import _run_intermediate_core
    SWEPT_CORES["late"] = (_run_late_core,
                           {"lr": "lr", "hidden": "text_hidden", "dropout": "text_dropout"})
    SWEPT_CORES["intermediate"] = (_run_intermediate_core,
                                   {"lr": "lr", "hidden": "hidden", "dropout": "dropout"})
except ImportError as exc:
    print("tuned wrappers unavailable:", exc)

def make_tuned(core_fn, hparam_names, cfg):
    """A (splits, emb, seed, device, encoder) callable pinned to `cfg`, returning
    (y_val, val_logits, y_test, test_logits) like the *_logits variants."""
    kwargs = {hparam_names[k]: v for k, v in cfg.items()}
    def _run(splits, emb, seed, device, encoder_name):
        r = core_fn(splits, emb, seed, device, encoder_name, **kwargs)
        return r["y_val"], r["val_logits"], r["y_test"], r["test_logits"]
    _run.__name__ = f"tuned_{core_fn.__name__}"
    return _run

# Methods that can hand back logits, needed for the post-hoc bias correction.
LOGIT_FNS = {}
try:
    from late import run_late_logits;                LOGIT_FNS["late"] = run_late_logits
    from inter import run_intermediate_logits;       LOGIT_FNS["intermediate"] = run_intermediate_logits
    from film import run_intermediate_film_logits;   LOGIT_FNS["intermediate_film"] = run_intermediate_film_logits
    from attn import run_intermediate_attn_logits;   LOGIT_FNS["intermediate_attn"] = run_intermediate_attn_logits
    from early import run_early_logits;              LOGIT_FNS["early"] = run_early_logits
    from text_ft import run_text_only_finetuned_logits; LOGIT_FNS["text_only_finetuned"] = run_text_only_finetuned_logits
    from report import run_emotion_only_logits, run_text_only_logits
    LOGIT_FNS["emotion_only"] = run_emotion_only_logits
    LOGIT_FNS["text_only"] = run_text_only_logits
except ImportError as exc:
    print("some *_logits variants unavailable:", exc)

for phase in RUN_PLAN:
    C.EMOTION_REGIME = phase["regime"]
    C.CLASS_WEIGHTING = phase["weighted"]
    C.CHECKPOINT_SINK = checkpoints        # collect trained models
    print(f"\n{'='*72}\nPHASE {phase['name']}  regime={phase['regime']}  "
          f"class_weighting={phase['weighted']}\n{'='*72}")

    for variant in phase["variants"]:
        splits = build_splits(variant)
        print(f"\n-- variant={variant}  train={len(splits['train'])} "
              f"val={len(splits['val'])} test={len(splits['test'])}")

        for encoder in ENCODERS:
            emb = get_embeddings(splits, encoder, DEVICE, cache_dir=CACHE_DIR)
            tok = None
            if "intermediate_attn" in phase["methods"]:
                # Token-level cache, separate from the pooled one. Both encoders
                # go through AutoModel here — that is what fixed the minilm NaN.
                tok = get_token_embeddings(splits, encoder, DEVICE, cache_dir=TOK_CACHE_DIR)

            for method in phase["methods"]:
                fn = ALL_METHODS[method]
                payload = tok if method == "intermediate_attn" else emb
                for seed in RUN_SEEDS:
                    tag = f"{phase['name']}/{variant}/{encoder}/{method}/seed{seed}"
                    try:
                        t0 = time.time()
                        cfg_key = (phase["weighted"], method, encoder)
                        if method in SWEPT_CORES and cfg_key in sweep_winners:
                            core, names = SWEPT_CORES[method]
                            lf = make_tuned(core, names,
                                            sweep_winners[cfg_key]["best_config"])
                        else:
                            lf = LOGIT_FNS.get(method)
                        if lf is not None:
                            y_val, val_lg, y_true, test_lg = lf(splits, payload, seed, DEVICE, encoder)
                            y_pred = np.argmax(test_lg, axis=1)
                            logits_by[(phase["name"], method, encoder, variant, seed)] = \
                                (y_val, val_lg, y_true, test_lg)
                        else:
                            y_true, y_pred = fn(splits, payload, seed, DEVICE, encoder)
                        assert y_true.shape == y_pred.shape == (len(splits["test"]),), "bad shape"
                        rec = C.make_result(method, encoder, variant, seed, y_true, y_pred)
                        rec["phase"] = phase["name"]
                        rec["regime"] = phase["regime"]
                        rec["class_weighting"] = phase["weighted"]
                        rec["tuned"] = bool(method in SWEPT_CORES and cfg_key in sweep_winners)
                        results.append(rec)
                        print(f"   {tag:62s} acc={rec['acc']:.4f} "
                              f"mF1={rec['macro_f1']:.4f} ({time.time()-t0:.0f}s)")
                    except Exception as exc:
                        failures.append((tag, repr(exc)))
                        print(f"   {tag:62s} FAILED: {exc}")
                        traceback.print_exc()

C.CHECKPOINT_SINK = None
print(f"\n{len(results)} runs ok, {len(failures)} failed, {time.time()-t_start:.0f}s total")
with open(f"{LOCAL_OUT}/results.json", "w") as fh:
    json.dump(results, fh, indent=2)

## 8. Post-hoc class-bias correction — over all 3 seeds

Fits three additive logit offsets on VAL, applies them to TEST. No retraining.

This is not a cosmetic step. In v1 it added **+0.10 macro-F1 to `late`** and **0.00 to
`intermediate_attn`**, which was enough to overturn the headline finding: uncorrected, `late` scored
below `emotion_only` in all four cells; corrected, it beat it in all four. A large part of "late
fusion destroys information" was really "late fusion's decision threshold is uncalibrated".

v1 reported this on a single seed. Here it runs on all three so the gain gets an error bar.

In [ ]:
import pandas as pd

bias_rows = []
for (phase, method, encoder, variant, seed), (y_val, val_lg, y_true, test_lg) in logits_by.items():
    b = tune_class_bias(val_lg, y_val)
    unc = C.compute_metrics(y_true, np.argmax(test_lg, axis=1))
    cor = C.compute_metrics(y_true, np.argmax(apply_class_bias(test_lg, b), axis=1))
    bias_rows.append(dict(phase=phase, method=method, encoder=encoder, variant=variant,
                          seed=seed, macro_f1_uncorrected=unc["macro_f1"],
                          macro_f1_corrected=cor["macro_f1"],
                          f1_borderline_uncorrected=unc["f1_borderline"],
                          f1_borderline_corrected=cor["f1_borderline"],
                          biases=[round(float(x), 3) for x in b]))

bias_df = pd.DataFrame(bias_rows)
bias_agg = (bias_df.groupby(["phase", "method", "encoder", "variant"])
            .agg(mf1_unc_mean=("macro_f1_uncorrected", "mean"),
                 mf1_unc_std=("macro_f1_uncorrected", "std"),
                 mf1_cor_mean=("macro_f1_corrected", "mean"),
                 mf1_cor_std=("macro_f1_corrected", "std"),
                 n_seeds=("seed", "count"))
            .reset_index())
bias_agg["gain"] = bias_agg.mf1_cor_mean - bias_agg.mf1_unc_mean
bias_agg = bias_agg.sort_values("gain", ascending=False)
bias_df.to_csv(f"{LOCAL_OUT}/bias_corrections_per_seed.csv", index=False)
bias_agg.to_csv(f"{LOCAL_OUT}/bias_corrections.csv", index=False)
print("who gains most from calibration (gain > its own std means it is real):")
bias_agg.round(4)

## 9. Results

In [ ]:
pd.set_option("display.width", 240, "display.max_columns", 60, "display.max_rows", 200)

# results_table() groups by (method, encoder, variant) only, so feeding it every
# phase at once would silently AVERAGE p1..p4 together. Always scope it to one
# phase; the phase-aware table below is the one to actually read.
MAIN_PHASE = RUN_PLAN[1]["name"] if len(RUN_PLAN) > 1 else RUN_PLAN[0]["name"]
main_results = [r for r in results if r["phase"] == MAIN_PHASE]
df = results_table(main_results)
print("plots/df scoped to phase:", MAIN_PHASE)
full = pd.DataFrame(results)
tbl = (full.groupby(["phase", "regime", "class_weighting", "method", "encoder", "variant"])
       .agg(acc_mean=("acc", "mean"), acc_std=("acc", "std"),
            macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
            f1_normal_mean=("f1_normal", "mean"),
            f1_borderline_mean=("f1_borderline", "mean"),
            f1_anomaly_mean=("f1_anomaly", "mean"),
            n_seeds=("seed", "count"))
       .reset_index()
       .sort_values(["phase", "variant", "encoder", "macro_f1_mean"], ascending=[1, 1, 1, 0]))
tbl.to_csv(f"{LOCAL_OUT}/results_table.csv", index=False)
tbl.round(4)

In [ ]:
# Headline: macro-F1 per method per phase, variant=full
piv = (tbl[tbl.variant == "full"]
       .pivot_table(index="method", columns=["phase", "encoder"], values="macro_f1_mean")
       .reindex(["majority", "text_only", "text_only_finetuned", "emotion_only",
                 "late", "intermediate", "intermediate_film", "intermediate_attn", "early"]))
print("macro-F1 (variant=full)")
piv.round(4)

In [ ]:
# The ablation ladder: what does each ingredient actually buy?
# frozen text -> fine-tuned text -> fine-tuned text + emotion at the input.
# In v1 the emotion token was worth ~2x the fine-tuning, which is the single
# number that defends "early fusion works because of fusion, not capacity".
for phase in tbl.phase.unique():
    sub = tbl[(tbl.phase == phase) & (tbl.variant == "full")]
    for enc in sub.encoder.unique():
        g = sub[sub.encoder == enc].set_index("method").macro_f1_mean
        if not {"text_only", "text_only_finetuned", "early"} <= set(g.index):
            continue
        t, tf, ea = g["text_only"], g["text_only_finetuned"], g["early"]
        ratio = (ea - tf) / (tf - t) if tf > t else float("nan")
        print(f"{phase:24s} {enc:6s}  frozen {t:.4f} -> +finetune {tf:.4f} "
              f"(D{tf-t:+.4f}) -> +emotion {ea:.4f} (D{ea-tf:+.4f})  "
              f"emotion/finetune = {ratio:.2f}x")

In [ ]:
# Deployment gap: oracle emotion vs SER-realistic emotion, and whether
# training under noise recovers it. Same test condition in p3 and p4.
gap = (tbl[(tbl.variant == "full") & (tbl.method.isin(EMOTION_DEPENDENT))]
       .pivot_table(index=["method", "encoder"], columns="phase", values="macro_f1_mean"))
cols = [c for c in ["p2_oracle_weighted", "p3_sertest_weighted", "p4_serboth_weighted"]
        if c in gap.columns]
if len(cols) >= 2:
    gap = gap[cols]
    if {"p3_sertest_weighted", "p4_serboth_weighted"} <= set(cols):
        gap["recovered"] = gap["p4_serboth_weighted"] - gap["p3_sertest_weighted"]
    if {"p2_oracle_weighted", "p3_sertest_weighted"} <= set(cols):
        gap["deployment_cost"] = gap["p3_sertest_weighted"] - gap["p2_oracle_weighted"]
print("oracle -> deployment -> trained-under-noise")
gap.round(4)

In [ ]:
plot_results(df, f"{LOCAL_OUT}/fusion_macro_f1.png")
from IPython.display import Image, display
display(Image(f"{LOCAL_OUT}/fusion_macro_f1.png"))

best = (tbl[(tbl.phase == MAIN_PHASE) & (tbl.variant == "full")
            & (tbl.method.str.contains("intermediate|early|late"))]
        .sort_values("macro_f1_mean", ascending=False).iloc[0])
print("best fusion:", best["method"], best["encoder"], best["phase"],
      f"macroF1={best['macro_f1_mean']:.4f} +/- {best['macro_f1_std']:.4f}")
plot_confusion([r for r in results if r["phase"] == best["phase"]],
               best["method"], best["encoder"], "full",
               f"{LOCAL_OUT}/confusion_best.png")
display(Image(f"{LOCAL_OUT}/confusion_best.png"))

## 10. Save to Drive

Layout:

```
CLEAR/fusion/
├── emb_cache/           pooled embeddings, reused across phases and sessions
├── emb_cache_tok/       token-level embeddings (intermediate_attn)
└── runs/<timestamp>/
    ├── results.json  results_table.csv
    ├── sweep_winners.json
    ├── bias_corrections.csv  bias_corrections_per_seed.csv
    ├── plots/
    └── checkpoints/
```

**Checkpoint policy.** Every small head is kept — they are kilobytes. Of the fine-tuned encoders
only the best val macro-F1 per (method, encoder) is kept: storing all of them would be ~48 x 440 MB
and the app needs one deployable model, not forty-eight.

In [ ]:
import shutil

stamp = time.strftime("%Y%m%d_%H%M")
run_dir = f"{RUNS_DIR}/{stamp}"
os.makedirs(f"{run_dir}/plots", exist_ok=True)
os.makedirs(f"{run_dir}/checkpoints", exist_ok=True)

for fname in ("results.json", "results_table.csv", "sweep_winners.json",
              "bias_corrections.csv", "bias_corrections_per_seed.csv"):
    src = f"{LOCAL_OUT}/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f"{run_dir}/{fname}")
for fname in os.listdir(LOCAL_OUT):
    if fname.endswith(".png"):
        shutil.copy(f"{LOCAL_OUT}/{fname}", f"{run_dir}/plots/{fname}")

# --- checkpoints ---------------------------------------------------------
# Small heads: keep all. Fine-tuned encoders: keep only the best per
# (method, encoder), because 48 x 440 MB is 21 GB of no use to anyone.
BIG = ("early_or_textft",)
best_big = {}
small = {}
for tag, payload in checkpoints.items():
    is_big = any(tag.startswith(p) for p in BIG)
    key = (payload["meta"].get("method"), payload["meta"].get("encoder"))
    if is_big:
        if key not in best_big or payload["val_f1"] > best_big[key][1]["val_f1"]:
            best_big[key] = (tag, payload)
    else:
        small[tag] = payload

saved = 0
for tag, payload in small.items():
    torch.save(payload, f"{run_dir}/checkpoints/{tag}.pt")
    saved += 1
for key, (tag, payload) in best_big.items():
    torch.save(payload, f"{run_dir}/checkpoints/BEST_{tag}.pt")
    saved += 1

manifest = dict(
    timestamp=stamp, device=DEVICE, dataset=DATASET_PATH,
    n_results=len(results), failures=failures,
    run_plan=[{k: v for k, v in p.items()} for p in RUN_PLAN],
    encoders=ENCODERS, seeds=RUN_SEEDS, widen_grid=WIDEN_GRID, pre_flight=PRE_FLIGHT,
    max_finetune_epochs=C.MAX_FINETUNE_EPOCHS,
    ser_noise_source=SER.SER_CONFUSION_SOURCE,
    ser_confusion={f"{k[0]}|{k[1]}": v for k, v in SER.SER_CONFUSION.items()},
    checkpoints_saved=saved,
)
with open(f"{run_dir}/manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2, default=str)

print("saved to", run_dir)
for root, _, files in os.walk(run_dir):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, run_dir):52s} {os.path.getsize(p)//1024:>8d} KB")

## 11. Reading the result honestly

Before any number goes on a slide:

1. **Is the gap bigger than the seed std?** Three seeds. If two methods differ by less than their
   pooled standard deviation, they are tied. v1's attn-vs-early gap was +0.011 against a pooled sd
   of 0.020 — that is a tie, not a win.
2. **Report the bias-corrected numbers, or report both.** Calibration moved `late` by +0.10 and
   `attn` by 0.00 in v1. A ranking computed on uncorrected logits is partly a ranking of how
   well-calibrated each method happens to be.
3. **Does anything beat `text_only_finetuned`?** That is the control for capacity. If `early` only
   matches it, the emotion channel is contributing nothing and that is the finding.
4. **Quote the deployment number, not just the oracle number.** Phase 3 is what Friday's live demo
   will actually look like. Phase 2 is an upper bound that no deployment can reach.
5. **Do not claim a winner between `early` and `intermediate`** — it flipped with the encoder in v1.
   The defensible claim is `{attn, early} > intermediate > late > single channel`.

### Limitations to state, not hide
- Symbolic-level fusion, not end-to-end audio.
- Labels are LLM-judge consensus; 46% of rows were labelled by a single judge.
- Hyperparameters were selected on val, with no separate selection split. Measured selection
  optimism in v1: `intermediate/minilm` scored 0.5702 on val and 0.5379 on test.
- `early` and `text_only_finetuned` were not swept, so their numbers are a lower bound.
- The sweep winners were found on the oracle regime and reused for the SER-noise phases.
- The SER noise model is calibrated at the **arousal** level (measured, n=902). The within-bucket
  emotion identity is assumed from the dataset prior — justified by the measurement that 6-way and
  2-way emotion features score identically (0.4111 both), so nothing label-relevant is lost.


## 12. Shut the runtime down

Only fires when `DISCONNECT_WHEN_DONE` is True, and only after verifying that the run actually
landed on Drive — disconnecting on a failed save would throw away the whole night. Leave it False
while you are working interactively.

Set it True together with `PRE_FLIGHT = "off"` before you go to bed.

In [ ]:
DISCONNECT_WHEN_DONE = False

def _preflight_ok():
    """Refuse to disconnect unless the run is genuinely finished and saved."""
    problems = []
    if PRE_FLIGHT != "off":
        problems.append(f"this was a pre-flight run (PRE_FLIGHT={PRE_FLIGHT!r}), not the real one")
    if failures:
        problems.append(f"{len(failures)} runs failed: {[t for t, _ in failures[:5]]}")
    if not results:
        problems.append("no results were produced")
    for required in ("results.json", "results_table.csv", "manifest.json"):
        if not os.path.exists(f"{run_dir}/{required}"):
            problems.append(f"missing on Drive: {required}")
    saved_ck = len(os.listdir(f"{run_dir}/checkpoints")) if os.path.isdir(f"{run_dir}/checkpoints") else 0
    if saved_ck == 0:
        problems.append("no checkpoints were written")
    return problems

problems = _preflight_ok()
print(f"run dir : {run_dir}")
print(f"results : {len(results)} ok, {len(failures)} failed")
print(f"best    : {best['method']}/{best['encoder']} macroF1={best['macro_f1_mean']:.4f}")

if problems:
    print()
    print("NOT disconnecting — unresolved problems:")
    for p in problems:
        print("  -", p)
elif not DISCONNECT_WHEN_DONE:
    print()
    print("Everything saved. DISCONNECT_WHEN_DONE is False, so the runtime stays up.")
else:
    # Flush Drive before pulling the plug: Colab's Drive mount is not
    # guaranteed to have finished writing just because shutil.copy returned.
    import subprocess, time as _t
    subprocess.run(["sync"], check=False)
    _t.sleep(20)
    print()
    print("Everything saved. Disconnecting the runtime in 10s...")
    _t.sleep(10)
    from google.colab import runtime
    runtime.unassign()